# Consolidated pipeline: RGB crop segmentation + Sequoia NDVI / NDRE

Part 1 (`01_rgb_segmentation_gli`) reads the **RGB dataset from Roboflow**, trains the lightweight CNN (AgriSegNet-Nano)
and compares it with YOLO26n-seg (both from scratch, FP32 + INT8, 640 / 480 / 320 px). Part 2 (`02_sequoia_alignment_indices`) reads the **12 Sequoia TIFs (4 captures x
NIR/RED/REG) from Google Drive**, calibrates and aligns them, and computes NDVI and NDRE. Results are saved to
`MyDrive/AgriEdgeNet_Results` and zipped at the end.

Before running: (1) Runtime -> Change runtime type -> GPU; (2) add the Colab Secret `ROBOFLOW_API_KEY` or fill
`ROBOFLOW_API_KEY` in the configuration cell; (3) make sure `MyDrive/parrot 12 images` exists (add a shortcut if the
folder was shared with you).

**What this notebook does not do:** it does not compute NDVI or NDRE per segmented crop patch. That needs the
RGB masks and the Sequoia bands on one shared pixel grid, which the current data cannot provide:

* the RGB frames come from a different camera (IMX219 parameters in the old notebook) than the Sequoia, with no
  shared georeference, and
* the 12 Sequoia frames (`IMG_260707_*`, 7 July 2026) show tree canopy over a road, not the five crops.

To get per-crop indices, either (a) fly the Sequoia over the crop plots and segment its own RGB or registered
bands, or (b) build orthomosaics of both sensors on a shared CRS and rasterise the masks onto the index grid.

# Part 1: RGB crop segmentation

# RGB crop segmentation with a device-agnostic lightweight CNN (AgriSegNet-Nano) + per-patch GLI and area

Replaces YOLO26n-seg with **AgriSegNet-Nano**: 34.7k parameters, 0.25 GMACs at 640 px, built only from
operators that every edge runtime supports. After BN folding the exported graph contains Conv
(dense + depthwise, no dilation), Add, Mul, Clip (ReLU6), GlobalAveragePool, Sigmoid and Resize with fixed
integer factors. It is static, with no Shape/Slice ops, so it runs on TensorRT (Jetson Nano/Orin), ONNX Runtime
(Raspberry Pi), OpenVINO, NPUs and, via TFLite-Micro, microcontrollers.

Flow: Roboflow export (train/valid/test, YOLO polygon labels) -> semantic masks -> train AgriSegNet-Nano and the
YOLO26n-seg baseline, both **from scratch**, same split / input size / epoch budget, both with **early stopping on
the valid split** -> YOLO's confidence threshold chosen on valid -> **deployment benchmark on test**: both models
exported to ONNX at 640 / 480 / 320 px, each in FP32 and INT8 (same quantisation recipe), scored with ONNX Runtime
(mIoU, per-class IoU, paired bootstrap CI, size, CPU latency) -> per-patch area (m2) and GLI drawn on the test images.

**Baseline.** YOLO26n-seg (3.13M params, 10.6 GFLOPs @640), the smallest YOLO26 segmentation model, trained from random
initialisation like Nano (34.7k params), so the comparison isolates architecture and size, not pretraining.

**On devices the binding limit is activation memory, not parameters.** INT8 peak activations are about 2.7 MiB at
640 px but 0.44 MiB at 256 px. The resolution sweep below shows the accuracy cost of smaller inputs.

Fixes carried over from the review (`docs/REVIEW.md`): the API key comes from Colab Secrets (**rotate the old
leaked key**); GSD is computed per image; GLI is reported over vegetation pixels; yield is labelled indicative;
leakage across splits is checked.

In [ ]:
# ---------------------------------------------------------------- setup
import os, sys, csv, json, subprocess, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow', 'onnx', 'onnxruntime', 'onnxscript'],
                   check=True)

import numpy as np
import cv2
import yaml
import torch

ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
if IN_COLAB:                       # results (incl. trained checkpoints) go to Drive so a disconnect loses nothing
    from google.colab import drive
    drive.mount('/content/drive')
RESULTS_ROOT = Path('/content/drive/MyDrive/AgriEdgeNet_Results') if IN_COLAB else ROOT / 'Results'
AGRISEG_DIR = Path(os.environ.get('AGRISEG_DIR', str(ROOT / 'agriseg')))

## agriseg package
Written to `/content/agriseg` by the cells below (generated from the repo; edit the repo, not these cells).

In [ ]:
!mkdir -p /content/agriseg

In [ ]:
%%writefile /content/agriseg/agrisegnet.py
"""AgriSegNet: lightweight semantic segmentation network for UAV crop mapping.

Design rationale (see README):
  * crop-type regions are large and boundaries are coarse -> spend capacity in the
    encoder / receptive field, predict at 1/8 resolution, bilinear-upsample.
  * ReLU6 only, no SE / hard-swish -> stable INT8 and Edge-TPU / TensorRT friendly.
  * depthwise-separable blocks everywhere except the first fused block.

Width (`width`) and depth (`depth`) multipliers let you sweep the Pareto front the way
the hogweed U-Net paper did with W and D.
"""
from __future__ import annotations

import math
from typing import Dict, List, Sequence, Union

import torch
import torch.nn as nn
import torch.nn.functional as F


def _c(ch: int, width: float, divisor: int = 8) -> int:
    """Scale channels, keep them a multiple of `divisor` (tensor-core / NPU friendly)."""
    v = max(divisor, int(ch * width + divisor / 2) // divisor * divisor)
    if v < 0.9 * ch * width:
        v += divisor
    return v


class ConvBNAct(nn.Sequential):
    def __init__(self, cin, cout, k=3, s=1, groups=1, act=True):
        pad = k // 2
        layers: List[nn.Module] = [
            nn.Conv2d(cin, cout, k, s, pad, groups=groups, bias=False),
            nn.BatchNorm2d(cout),
        ]
        if act:
            layers.append(nn.ReLU6(inplace=True))
        super().__init__(*layers)


class FusedMBConv(nn.Module):
    """3x3 conv expand -> 1x1 linear project. Used at high resolution where depthwise
    convs are memory-bound and slow on GPUs."""

    def __init__(self, cin, cout, s=1, expand=2):
        super().__init__()
        mid = cin * expand
        self.use_res = s == 1 and cin == cout
        self.block = nn.Sequential(
            ConvBNAct(cin, mid, 3, s),
            ConvBNAct(mid, cout, 1, 1, act=False),
        )

    def forward(self, x):
        y = self.block(x)
        return x + y if self.use_res else y


class InvertedResidual(nn.Module):
    def __init__(self, cin, cout, s=1, expand=3, k=3):
        super().__init__()
        mid = cin * expand
        self.use_res = s == 1 and cin == cout
        layers: List[nn.Module] = []
        if expand != 1:
            layers.append(ConvBNAct(cin, mid, 1, 1))
        layers += [
            ConvBNAct(mid, mid, k, s, groups=mid),
            ConvBNAct(mid, cout, 1, 1, act=False),
        ]
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        y = self.block(x)
        return x + y if self.use_res else y


class DWSep(nn.Sequential):
    """depthwise 3x3 + pointwise 1x1 (both with BN, ReLU6 after the pointwise)."""

    def __init__(self, cin, cout):
        super().__init__(
            ConvBNAct(cin, cin, 3, 1, groups=cin),
            ConvBNAct(cin, cout, 1, 1),
        )


class ContextModule(nn.Module):
    """Tiny pyramid pooling: injects global / regional context (several crop rows to a
    whole plot) at 1/32 resolution where it is almost free."""

    def __init__(self, cin, cout, bins: Sequence[int] = (1, 2, 4)):
        super().__init__()
        mid = max(8, cout // 4)
        self.bins = tuple(bins)
        self.reduce = nn.ModuleList([ConvBNAct(cin, mid, 1, 1) for _ in bins])
        self.fuse = ConvBNAct(cin + mid * len(bins), cout, 1, 1)

    def forward(self, x):
        h, w = x.shape[-2:]
        outs = [x]
        for b, red in zip(self.bins, self.reduce):
            p = F.adaptive_avg_pool2d(x, b)
            outs.append(F.interpolate(red(p), size=(h, w), mode="bilinear", align_corners=False))
        return self.fuse(torch.cat(outs, dim=1))


class AgriSegNet(nn.Module):
    """
    Input : (B, 3, H, W), H and W divisible by 32; use multiples of 128 for a clean ONNX/TensorRT graph (the eval code pads for you)
    Output: (B, num_classes, H, W) logits.  In training mode with aux=True returns a dict
            {"out": ..., "aux": ...}; the aux head is dropped at inference/export.
    """

    def __init__(self, num_classes: int, width: float = 1.0, depth: int = 1,
                 decoder_ch: int = 64, aux: bool = True, drop: float = 0.1):
        super().__init__()
        self.num_classes = num_classes
        self.use_aux = aux
        c0, c1, c2, c3, c4 = (_c(c, width) for c in (16, 24, 48, 96, 160))
        d = _c(decoder_ch, width)

        self.stem = ConvBNAct(3, c0, 3, 2)                                   # 1/2
        s1 = [FusedMBConv(c0, c1, s=2, expand=2)]                            # 1/4
        s1 += [FusedMBConv(c1, c1, s=1, expand=2) for _ in range(depth - 1)]
        self.stage1 = nn.Sequential(*s1)
        self.stage2 = nn.Sequential(                                         # 1/8
            InvertedResidual(c1, c2, 2, 3, 3),
            *[InvertedResidual(c2, c2, 1, 3, 3) for _ in range(depth)])
        self.stage3 = nn.Sequential(                                         # 1/16
            InvertedResidual(c2, c3, 2, 3, 5),
            *[InvertedResidual(c3, c3, 1, 3, 5) for _ in range(depth + 1)])
        self.stage4 = nn.Sequential(                                         # 1/32
            InvertedResidual(c3, c4, 2, 3, 5),
            *[InvertedResidual(c4, c4, 1, 3, 5) for _ in range(depth)])

        self.context = ContextModule(c4, c3)
        self.lat16 = ConvBNAct(c3, d, 1, 1, act=False)
        self.lat8 = ConvBNAct(c2, d, 1, 1, act=False)
        self.ctx_proj = ConvBNAct(c3, d, 1, 1, act=False)
        self.dec16 = DWSep(d, d)
        self.dec8 = DWSep(d, d)
        self.drop = nn.Dropout2d(drop)
        self.head = nn.Conv2d(d, num_classes, 1)
        self.aux_head = nn.Sequential(ConvBNAct(c3, d, 1, 1), nn.Conv2d(d, num_classes, 1)) if aux else None
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x) -> Union[torch.Tensor, Dict[str, torch.Tensor]]:
        H, W = x.shape[-2:]
        x = self.stem(x)
        x = self.stage1(x)
        f8 = self.stage2(x)
        f16 = self.stage3(f8)
        f32 = self.stage4(f16)

        ctx = self.ctx_proj(self.context(f32))
        up = lambda t, ref: F.interpolate(t, size=ref.shape[-2:], mode="bilinear", align_corners=False)
        d16 = self.dec16(up(ctx, f16) + self.lat16(f16))
        d8 = self.dec8(up(d16, f8) + self.lat8(f8))
        out = self.head(self.drop(d8))
        out = F.interpolate(out, size=(H, W), mode="bilinear", align_corners=False)

        if self.training and self.use_aux:
            aux = F.interpolate(self.aux_head(f16), size=(H, W), mode="bilinear", align_corners=False)
            return {"out": out, "aux": aux}
        return out


def count_params(model: nn.Module, include_aux: bool = False) -> int:
    n = 0
    for name, p in model.named_parameters():
        if not include_aux and name.startswith("aux_head"):
            continue
        if p.requires_grad:
            n += p.numel()
    return n


def count_flops(model: nn.Module, imgsz=(512, 512), device="cpu") -> float:
    """Returns GMACs for one forward pass (aux head excluded because eval() drops it)."""
    from torch.utils.flop_counter import FlopCounterMode
    model = model.eval().to(device)
    x = torch.zeros(1, 3, *imgsz, device=device)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        model(x)
    return fc.get_total_flops() / 2 / 1e9   # FLOPs -> MACs -> G


# ----------------------------------------------------------------------------- sub-50k variant
class DWBlock(nn.Module):
    """Depthwise 3x3 + pointwise 1x1, residual when shapes match. Stride 2 downsamples."""

    def __init__(self, cin, cout, s=1):
        super().__init__()
        self.use_res = s == 1 and cin == cout
        self.dw = ConvBNAct(cin, cin, 3, s, groups=cin)
        self.pw = ConvBNAct(cin, cout, 1, 1, act=not self.use_res)

    def forward(self, x):
        y = self.pw(self.dw(x))
        return x + y if self.use_res else y


class AgriSegNetNano(nn.Module):
    """Device-agnostic < 50k-parameter semantic segmentation network (crop-type mapping, UAV RGB).

    Operator set (after BN folding): Conv (dense + depthwise, no dilation), Add, Mul, Clip (ReLU6),
    GlobalAveragePool, Sigmoid, Resize (bilinear, integer factors). These map to TensorRT, ONNX Runtime,
    TFLite / TFLite-Micro, OpenVINO and most NPUs; nothing dynamic, no dilation, no attention.

    Capacity goes to receptive field (stages down to 1/32 + a global-context gate) rather than channels:
    crop-type regions are large and colour/texture driven. Prediction at 1/8.

    `output_stride=8` in forward() skips the final upsample: the deployed graph then emits logits at 1/8
    resolution (argmax there, nearest-upsample the label map on the host). At 640 px this removes the
    largest activation in the network (640*640*num_classes).
    """

    def __init__(self, num_classes: int, width: float = 1.0, aux: bool = True, drop: float = 0.05):
        super().__init__()
        self.num_classes, self.use_aux = num_classes, aux
        self.output_stride = 1
        c0, c1, c2, c3 = (_c(c, width) for c in (16, 24, 32, 48))
        self.stem = ConvBNAct(3, c0, 3, 2)                                    # 1/2
        self.stage4 = nn.Sequential(DWBlock(c0, c1, 2), InvertedResidual(c1, c1, 1, 2, 3))   # 1/4
        self.stage8 = nn.Sequential(InvertedResidual(c1, c2, 2, 2, 3),         # 1/8
                                    InvertedResidual(c2, c2, 1, 2, 3))
        self.stage16 = nn.Sequential(InvertedResidual(c2, c3, 2, 2, 3), DWBlock(c3, c3))   # 1/16
        self.stage32 = nn.Sequential(DWBlock(c3, c3, 2), DWBlock(c3, c3))      # 1/32
        self.ctx16 = ConvBNAct(c3, c2, 1, 1)
        self.ctx32 = ConvBNAct(c3, c2, 1, 1, act=False)
        self.gate = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c3, c2, 1), nn.Sigmoid())
        self.low = ConvBNAct(c2, c2, 1, 1, act=False)
        self.refine = DWBlock(c2, c2)
        self.drop = nn.Dropout2d(drop)
        self.head = nn.Conv2d(c2, num_classes, 1)
        self.aux_head = nn.Conv2d(c3, num_classes, 1) if aux else None
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        # input H, W must be divisible by 32; fixed integer scale factors keep the exported graph static
        # (no Shape/Slice/Concat ops), which microcontroller and NPU compilers require
        f8 = self.stage8(self.stage4(self.stem(x)))
        f16 = self.stage16(f8)
        f32 = self.stage32(f16)
        up = lambda t, k: F.interpolate(t, scale_factor=k, mode="bilinear", align_corners=False)
        ctx = self.ctx16(f16) * self.gate(f32) + up(self.ctx32(f32), 2)
        out = self.head(self.drop(self.refine(up(ctx, 2) + self.low(f8))))
        if self.output_stride == 1:
            out = up(out, 8)
        if self.training and self.use_aux:
            return {"out": out, "aux": up(self.aux_head(f16), 16)}
        return out


def peak_activation_bytes(model: nn.Module, imgsz=(640, 640), bytes_per_value=1) -> int:
    """Largest (input + output) of any conv, plus the final upsample if present. BN and activations are
    folded or in-place at deployment, so they are not counted. A lower bound on the runtime arena size."""
    peak = [0]

    def hook(_m, inp, out):
        peak[0] = max(peak[0], out.numel() + inp[0].numel())

    hs = [m.register_forward_hook(hook) for m in model.modules() if isinstance(m, nn.Conv2d)]
    with torch.no_grad():
        out = model.eval()(torch.zeros(1, 3, *imgsz))
    for h in hs:
        h.remove()
    if getattr(model, "output_stride", 1) == 1:      # final bilinear upsample: 1/8 logits -> full res
        peak[0] = max(peak[0], out.numel() + out.numel() // 64)
    return peak[0] * bytes_per_value


ARCHS = {"agrisegnet": AgriSegNet, "nano": AgriSegNetNano}
PARAM_BUDGET = {"agrisegnet": 1_000_000, "nano": 50_000}


def build_model(arch: str, num_classes: int, width: float = 1.0, depth: int = 1, aux: bool = True) -> nn.Module:
    if arch == "nano":
        return AgriSegNetNano(num_classes, width=width, aux=aux)
    return AgriSegNet(num_classes, width=width, depth=depth, aux=aux)


if __name__ == "__main__":
    print(f"{'arch':<11}{'width':>6}{'params':>10}{'GMACs@640':>11}   peak INT8 activations (MiB) @256 / 320 / 640")
    for arch, w in [("nano", 0.5), ("nano", 0.75), ("nano", 1.0), ("agrisegnet", 0.25), ("agrisegnet", 1.0)]:
        m = build_model(arch, num_classes=6, width=w, aux=False)
        acts = [peak_activation_bytes(m, (s, s)) / 2 ** 20 for s in (256, 320, 640)]
        line = f"{arch:<11}{w:>6}{count_params(m):>10,d}{count_flops(m, (640, 640)):>11.3f}   " + " / ".join(f"{a:.2f}" for a in acts)
        if arch == "nano":
            m.output_stride = 8
            line += "   | stride-8 output @640: %.2f" % (peak_activation_bytes(m, (640, 640)) / 2 ** 20)
        print(line)

In [ ]:
%%writefile /content/agriseg/dataset.py
"""Roboflow / Ultralytics YOLO-segmentation export  ->  semantic segmentation dataset.

Expected layout (what Roboflow's "YOLOv8/YOLO26 segmentation" export gives you):

    <root>/data.yaml            (names: [...])
    <root>/train/images/*.jpg   <root>/train/labels/*.txt
    <root>/valid/images/*.jpg   <root>/valid/labels/*.txt     ('val' also accepted)
    <root>/test/images/*.jpg    <root>/test/labels/*.txt

Each label line:  class_id x1 y1 x2 y2 ...   (normalised polygon)
Semantic mask   :  0 = background, 1..nc = class_id + 1, 255 = ignore (padding only)
"""
from __future__ import annotations

import random
from pathlib import Path
from typing import List, Tuple

import cv2
import numpy as np
import torch
import yaml
from torch.utils.data import Dataset

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32) * 255
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32) * 255
IGNORE = 255
PAD_MULT = 32    # network stride; the onnx exporter rounds agrisegnet inputs to 128 (context-pool bins 1,2,4)


def load_names(root: Path) -> List[str]:
    cfg = yaml.safe_load(open(Path(root) / "data.yaml"))
    names = cfg["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    return list(names)


def find_split_dir(root: Path, split: str) -> Path:
    aliases = {"val": ["valid", "val"], "valid": ["valid", "val"]}.get(split, [split])
    for a in aliases:
        if (Path(root) / a / "images").is_dir():
            return Path(root) / a
    raise FileNotFoundError(f"No '{split}' split under {root} (looked for {aliases})")


def parse_label_file(path: Path) -> List[Tuple[int, np.ndarray]]:
    """-> list of (class_id, polygon[N,2] normalised). Bounding boxes (5 values) become rectangles."""
    out = []
    if not path.exists():
        return out
    for line in path.read_text().strip().splitlines():
        v = line.split()
        if len(v) < 5:
            continue
        cls = int(float(v[0]))
        vals = np.array(v[1:], dtype=np.float64)
        if len(vals) == 4:                       # cx cy w h  -> rectangle
            cx, cy, w, h = vals
            poly = np.array([[cx - w / 2, cy - h / 2], [cx + w / 2, cy - h / 2],
                             [cx + w / 2, cy + h / 2], [cx - w / 2, cy + h / 2]])
        else:
            if len(vals) % 2:
                vals = vals[:-1]
            poly = vals.reshape(-1, 2)
        if len(poly) >= 3:
            out.append((cls, poly))
    return out


def rasterize(polys: List[Tuple[int, np.ndarray]], h: int, w: int) -> np.ndarray:
    mask = np.zeros((h, w), np.uint8)
    for cls, poly in polys:                       # later polygons overwrite earlier ones
        pts = np.round(poly * np.array([w, h])).astype(np.int32)
        cv2.fillPoly(mask, [pts], int(cls) + 1)
    return mask


def class_frequencies(label_dir: Path, nc: int) -> np.ndarray:
    """Approximate pixel share of [background, class0..] from polygon areas (no image reads)."""
    areas = np.zeros(nc + 1)
    for f in Path(label_dir).glob("*.txt"):
        tot = 0.0
        for cls, p in parse_label_file(f):
            x, y = p[:, 0], p[:, 1]
            a = 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))
            areas[cls + 1] += a
            tot += a
        areas[0] += max(0.0, 1.0 - tot)
    return areas / max(areas.sum(), 1e-9)


# ----------------------------------------------------------------------------- augmentation
def _color_jitter(img: np.ndarray, rng: random.Random) -> np.ndarray:
    img = img.astype(np.float32)
    img = img * rng.uniform(0.7, 1.3) + rng.uniform(-25, 25)                    # contrast / brightness
    if rng.random() < 0.5:                                                      # gamma
        g = rng.uniform(0.7, 1.4)
        img = 255 * (np.clip(img, 0, 255) / 255) ** g
    hsv = cv2.cvtColor(np.clip(img, 0, 255).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[..., 0] = (hsv[..., 0] + rng.uniform(-6, 6)) % 180                      # hue
    hsv[..., 1] *= rng.uniform(0.7, 1.3)                                        # saturation
    img = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2RGB)
    if rng.random() < 0.2:                                                      # motion-blur proxy
        k = rng.choice([3, 5, 7])
        kern = np.zeros((k, k), np.float32)
        kern[k // 2, :] = 1.0 / k
        ang = rng.uniform(0, 180)
        M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), ang, 1.0)
        kern = cv2.warpAffine(kern, M, (k, k))
        kern /= max(kern.sum(), 1e-6)
        img = cv2.filter2D(img, -1, kern)
    if rng.random() < 0.2:                                                      # sensor noise
        img = np.clip(img.astype(np.float32) + np.random.normal(0, rng.uniform(2, 8), img.shape), 0, 255).astype(np.uint8)
    return img


def _affine(img, mask, out: int, rng: random.Random, scale_rng=(0.6, 1.6)):
    """Random scale / rotation (any heading) / flip / translation in one warp."""
    h, w = mask.shape
    base = out / max(h, w)
    s = base * rng.uniform(*scale_rng)
    ang = rng.choice([0, 90, 180, 270]) + rng.uniform(-20, 20) if rng.random() < 0.8 else 0.0
    cx, cy = rng.uniform(0.2, 0.8) * w, rng.uniform(0.2, 0.8) * h
    M = cv2.getRotationMatrix2D((cx, cy), ang, s)
    M[0, 2] += out / 2 - cx
    M[1, 2] += out / 2 - cy
    if rng.random() < 0.5:                       # horizontal flip of the output
        M = np.array([[-1, 0, out - 1], [0, 1, 0]], np.float64) @ np.vstack([M, [0, 0, 1]])
    img = cv2.warpAffine(img, M[:2], (out, out), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    mask = cv2.warpAffine(mask, M[:2], (out, out), flags=cv2.INTER_NEAREST,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=IGNORE)
    return img, mask


def to_tensor(img_rgb: np.ndarray) -> torch.Tensor:
    x = (img_rgb.astype(np.float32) - MEAN) / STD
    return torch.from_numpy(x.transpose(2, 0, 1).copy())


# ----------------------------------------------------------------------------- datasets
class CropSegDataset(Dataset):
    """Training dataset: random affine + colour aug, fixed square output."""

    def __init__(self, root, split="train", imgsz=512, train=True, seed=0):
        self.root, self.imgsz, self.train = Path(root), imgsz, train
        d = find_split_dir(self.root, split)
        self.images = sorted(p for p in (d / "images").iterdir() if p.suffix.lower() in IMG_EXT)
        self.label_dir = d / "labels"
        self.names = load_names(self.root)
        self.nc = len(self.names)
        self.rng = random.Random(seed)

    def __len__(self):
        return len(self.images)

    def load(self, i):
        p = self.images[i]
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        mask = rasterize(parse_label_file(self.label_dir / (p.stem + ".txt")), h, w)
        return img, mask

    def __getitem__(self, i):
        img, mask = self.load(i)
        rng = self.rng if self.train else random.Random(i)
        img, mask = _affine(img, mask, self.imgsz, rng)
        img = _color_jitter(img, rng)
        return to_tensor(img), torch.from_numpy(mask.astype(np.int64))


def preprocess(img_rgb: np.ndarray, imgsz: int, pad_mult: int = PAD_MULT):
    """Aspect-preserving resize (long side = imgsz), reflect-pad bottom/right to a multiple of pad_mult.
    Returns (CHW float tensor, (valid_h, valid_w)). Used for evaluation AND deployment, so both see the same input."""
    h, w = img_rgb.shape[:2]
    s = imgsz / max(h, w)
    nh, nw = max(32, round(h * s)), max(32, round(w * s))
    small = cv2.resize(img_rgb, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    small = cv2.copyMakeBorder(small, 0, (-nh) % pad_mult, 0, (-nw) % pad_mult, cv2.BORDER_REFLECT_101)
    return to_tensor(small), (nh, nw)


class CropSegEval(Dataset):
    """Evaluation dataset: aspect-preserving resize (long side = imgsz), pad to a multiple of 128.
    Returns the ORIGINAL-resolution mask so every model is scored at the same resolution."""

    def __init__(self, root, split="test", imgsz=512):
        self.root, self.imgsz = Path(root), imgsz
        d = find_split_dir(self.root, split)
        self.images = sorted(p for p in (d / "images").iterdir() if p.suffix.lower() in IMG_EXT)
        self.label_dir = d / "labels"
        self.names = load_names(self.root)
        self.nc = len(self.names)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        p = self.images[i]
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        gt = rasterize(parse_label_file(self.label_dir / (p.stem + ".txt")), h, w)
        x, (nh, nw) = preprocess(img, self.imgsz)
        return {"image": x, "gt": torch.from_numpy(gt.astype(np.int64)),
                "valid_hw": (nh, nw), "path": str(p), "rgb": img}


def eval_collate(batch):          # batch size is 1 for eval; keep python objects intact
    assert len(batch) == 1
    b = batch[0]
    b["image"] = b["image"].unsqueeze(0)
    return b

In [ ]:
%%writefile /content/agriseg/metrics.py
"""Confusion-matrix based segmentation metrics + semantic conversion of YOLO instance masks."""
from __future__ import annotations

import cv2
import numpy as np


class ConfusionMatrix:
    def __init__(self, n):
        self.n = n
        self.m = np.zeros((n, n), np.int64)

    def update(self, gt: np.ndarray, pred: np.ndarray):
        k = gt != 255
        idx = self.n * gt[k].astype(np.int64) + pred[k].astype(np.int64)
        self.m += np.bincount(idx, minlength=self.n ** 2).reshape(self.n, self.n)

    def summary(self, names):
        tp = np.diag(self.m).astype(np.float64)
        gt_sum, pr_sum = self.m.sum(1), self.m.sum(0)
        union = gt_sum + pr_sum - tp
        iou = np.where(union > 0, tp / np.maximum(union, 1), np.nan)
        f1 = np.where(gt_sum + pr_sum > 0, 2 * tp / np.maximum(gt_sum + pr_sum, 1), np.nan)
        return {
            "pixel_acc": float(tp.sum() / max(self.m.sum(), 1)),
            "mIoU_all": float(np.nanmean(iou)),                       # incl. background
            "mIoU_crops": float(np.nanmean(iou[1:])),                 # crop classes only (headline number)
            "mF1_crops": float(np.nanmean(f1[1:])),
            "per_class_iou": {n: (None if np.isnan(v) else float(v)) for n, v in zip(names, iou)},
        }


def instances_to_semantic(polys_px, classes, confs, h, w):
    """YOLO instance masks -> one semantic map. Low-confidence instances are painted first so
    higher-confidence ones win overlaps. polys_px: list of Nx2 pixel polygons (original image
    coordinates). classes are dataset class ids (0-based); output is 0=bg, class+1."""
    out = np.zeros((h, w), np.uint8)
    for i in np.argsort(np.asarray(confs)):
        pts = np.round(np.asarray(polys_px[i])).astype(np.int32)
        if len(pts) >= 3:
            cv2.fillPoly(out, [pts], int(classes[i]) + 1)
    return out

In [ ]:
%%writefile /content/agriseg/engine.py
"""Shared inference / evaluation helpers (used by train.py and evaluate.py)."""
from __future__ import annotations

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from agrisegnet import build_model
from dataset import CropSegEval, eval_collate, preprocess
from metrics import ConfusionMatrix


def build_from_ckpt(path, device="cpu"):
    ck = torch.load(path, map_location=device, weights_only=False)
    cfg = ck["cfg"]
    model = build_model(cfg.get("arch", "agrisegnet"), cfg["num_classes"], width=cfg["width"],
                        depth=cfg["depth"], aux=False)
    sd = {k: v for k, v in ck["model"].items() if not k.startswith("aux_head")}
    model.load_state_dict(sd)
    return model.to(device).eval(), ck


@torch.no_grad()
def predict_semantic(model, batch, device, amp=False, tta=False) -> np.ndarray:
    """Returns uint8 map at ORIGINAL image resolution (0=bg, c+1)."""
    x = batch["image"].to(device, non_blocking=True)
    nh, nw = batch["valid_hw"]
    h, w = batch["gt"].shape
    with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
        logits = model(x).float()
        if tta:
            logits = logits.softmax(1)
            logits = logits + model(x.flip(3)).float().softmax(1).flip(3)
            logits = logits + model(x.flip(2)).float().softmax(1).flip(2)
    logits = logits[:, :, :nh, :nw]
    logits = F.interpolate(logits, size=(h, w), mode="bilinear", align_corners=False)
    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8)


@torch.no_grad()
def predict_image(model, img_rgb, imgsz, device=torch.device("cpu")) -> np.ndarray:
    """Label map (0=bg, c+1) at the image's own resolution, for any RGB image."""
    x, (nh, nw) = preprocess(img_rgb, imgsz)
    logits = model(x.unsqueeze(0).to(device)).float()
    logits = logits[:, :, :nh, :nw] if logits.shape[-2:] == x.shape[-2:] else \
        F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, :, :nh, :nw]
    h, w = img_rgb.shape[:2]
    logits = F.interpolate(logits, size=(h, w), mode="bilinear", align_corners=False)
    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8)


@torch.no_grad()
def evaluate_model(model, root, split, imgsz, device, names, amp=False, tta=False, workers=2,
                   max_images=None, save_pred_dir=None):
    model.eval()
    ds = CropSegEval(root, split, imgsz)
    n_cls = ds.nc + 1
    cm = ConfusionMatrix(n_cls)
    dl = DataLoader(ds, batch_size=1, shuffle=False, num_workers=workers, collate_fn=eval_collate)
    for i, batch in enumerate(dl):
        if max_images and i >= max_images:
            break
        pred = predict_semantic(model, batch, device, amp, tta)
        gt = batch["gt"].numpy().astype(np.uint8)
        cm.update(gt, pred)
        if save_pred_dir is not None:
            from pathlib import Path
            import cv2
            Path(save_pred_dir).mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(Path(save_pred_dir) / (Path(batch["path"]).stem + ".png")), pred)
    return cm.summary(["background"] + list(names)), cm

In [ ]:
%%writefile /content/agriseg/train.py
"""Train AgriSegNet on a Roboflow YOLO-segmentation export.

    python train.py --data /path/to/roboflow_export --imgsz 640 --epochs 150 --bs 16 \
                    --width 1.0 --depth 1 --seed 0 --out runs/agri_w1d1_s0

Use the SAME --imgsz you trained YOLO26 with, otherwise the comparison is not like-for-like.
"""
from __future__ import annotations

import argparse
import copy
import csv
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from agrisegnet import PARAM_BUDGET, build_model, count_flops, count_params
from dataset import CropSegDataset, class_frequencies, find_split_dir, load_names
from engine import evaluate_model


class ModelEMA:
    def __init__(self, model, decay=0.999, tau=1000):
        self.ema = copy.deepcopy(model).eval()
        self.decay, self.tau, self.updates = decay, tau, 0
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = self.decay * (1 - math.exp(-self.updates / self.tau))
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])


def dice_loss(logits, target, n_cls, ignore=255, eps=1.0):
    valid = (target != ignore)
    probs = logits.softmax(1) * valid.unsqueeze(1)
    t = target.clone()
    t[~valid] = 0
    onehot = F.one_hot(t, n_cls).permute(0, 3, 1, 2).float() * valid.unsqueeze(1)
    inter = (probs * onehot).sum((0, 2, 3))
    denom = probs.sum((0, 2, 3)) + onehot.sum((0, 2, 3))
    dice = (2 * inter + eps) / (denom + eps)
    present = onehot.sum((0, 2, 3)) > 0
    return 1 - dice[present].mean() if present.any() else logits.sum() * 0


def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def worker_init(base_seed):
    # deterministic per worker (the previous version mixed in wall-clock time, so --seed did not
    # reproduce a run)
    def _init(wid):
        info = torch.utils.data.get_worker_info()
        s = base_seed * 100_003 + 1000 * wid
        info.dataset.rng = random.Random(s)
        np.random.seed(s % (2 ** 32))
    return _init


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="runs/agrisegnet")
    ap.add_argument("--imgsz", type=int, default=512)
    ap.add_argument("--epochs", type=int, default=150)
    ap.add_argument("--bs", type=int, default=16)
    ap.add_argument("--lr", type=float, default=2e-3)
    ap.add_argument("--wd", type=float, default=1e-2)
    ap.add_argument("--warmup", type=int, default=3)
    ap.add_argument("--arch", choices=["agrisegnet", "nano"], default="agrisegnet")
    ap.add_argument("--max-params", type=int, default=None, help="default: 50k for nano, 1M for agrisegnet")
    ap.add_argument("--width", type=float, default=1.0)
    ap.add_argument("--depth", type=int, default=1)
    ap.add_argument("--aux-w", type=float, default=0.4)
    ap.add_argument("--dice-w", type=float, default=1.0)
    ap.add_argument("--val-every", type=int, default=1)
    ap.add_argument("--patience", type=int, default=0,
                    help="stop when val mIoU(crops) has not improved for this many epochs (0 = off)")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--no-amp", action="store_true")
    ap.add_argument("--max-val-images", type=int, default=None)
    a = ap.parse_args()

    seed_everything(a.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = (dev.type == "cuda") and not a.no_amp
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)

    names = load_names(Path(a.data))
    n_cls = len(names) + 1
    ds = CropSegDataset(a.data, "train", a.imgsz, train=True, seed=a.seed)
    dl = DataLoader(ds, batch_size=a.bs, shuffle=True, num_workers=a.workers, drop_last=len(ds) >= a.bs,
                    pin_memory=dev.type == "cuda", persistent_workers=a.workers > 0,
                    worker_init_fn=worker_init(a.seed))

    freq = class_frequencies(find_split_dir(Path(a.data), "train") / "labels", len(names))
    cw = 1.0 / np.sqrt(np.maximum(freq, 1e-3))
    cw = np.clip(cw / cw.mean(), 0.25, 4.0)
    print("class pixel share:", dict(zip(["background"] + names, np.round(freq, 3))))
    print("class weights    :", dict(zip(["background"] + names, np.round(cw, 2))))
    ce = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=dev), ignore_index=255)

    model = build_model(a.arch, n_cls, width=a.width, depth=a.depth, aux=a.aux_w > 0).to(dev)
    model = model.to(memory_format=torch.channels_last)
    n_params = count_params(model)
    gmacs = count_flops(build_model(a.arch, n_cls, a.width, a.depth, aux=False), (a.imgsz, a.imgsz))
    budget = a.max_params or PARAM_BUDGET[a.arch]
    print(f"{a.arch} w={a.width} d={a.depth}: {n_params:,} params (budget {budget:,}), {gmacs:.3f} GMACs @ {a.imgsz}")
    if n_params >= budget:
        raise SystemExit(f"{n_params:,} parameters exceeds the {budget:,} budget")

    decay, no_decay = [], []
    for n_, p in model.named_parameters():
        (no_decay if p.ndim <= 1 else decay).append(p)
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": a.wd},
                             {"params": no_decay, "weight_decay": 0.0}], lr=a.lr)
    steps_per_epoch = len(dl)
    total, warm = a.epochs * steps_per_epoch, a.warmup * steps_per_epoch
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    ema = ModelEMA(model)

    cfg = {"arch": a.arch, "num_classes": n_cls, "width": a.width, "depth": a.depth, "imgsz": a.imgsz, "names": names}
    log = open(out / "log.csv", "w", newline="")
    wr = csv.writer(log)
    wr.writerow(["epoch", "loss", "val_mIoU_crops", "val_mIoU_all", "val_pixel_acc", "lr", "sec"])
    best, best_ep, t0 = -1.0, -1, time.time()

    for ep in range(a.epochs):
        model.train()
        run, nb = 0.0, 0
        for x, y in dl:
            x = x.to(dev, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(dev, non_blocking=True)
            with torch.autocast(device_type=dev.type, enabled=amp):
                o = model(x)
                main_o = (o["out"] if isinstance(o, dict) else o).float()   # tensor when --aux-w 0
                loss = ce(main_o, y) + a.dice_w * dice_loss(main_o, y, n_cls)
                if isinstance(o, dict):
                    aux_o = o["aux"].float()
                    loss = loss + a.aux_w * (ce(aux_o, y) + a.dice_w * dice_loss(aux_o, y, n_cls))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step(); ema.update(model)
            run += loss.item(); nb += 1

        miou = miou_all = pacc = float("nan")
        if (ep + 1) % a.val_every == 0 or ep == a.epochs - 1:
            res, _ = evaluate_model(ema.ema.to(dev), a.data, "valid", a.imgsz, dev, names, amp=amp,
                                    workers=min(2, a.workers), max_images=a.max_val_images)
            miou, miou_all, pacc = res["mIoU_crops"], res["mIoU_all"], res["pixel_acc"]
            if miou > best:
                best, best_ep = miou, ep
                torch.save({"model": ema.ema.state_dict(), "cfg": cfg, "epoch": ep, "val": res}, out / "best.pt")
        torch.save({"model": ema.ema.state_dict(), "cfg": cfg, "epoch": ep}, out / "last.pt")
        wr.writerow([ep, run / max(nb, 1), miou, miou_all, pacc, opt.param_groups[0]["lr"], time.time() - t0]); log.flush()
        print(f"ep {ep+1:3d}/{a.epochs} loss {run/max(nb,1):.4f} val mIoU(crops) {miou:.4f} best {best:.4f}", flush=True)
        if a.patience and ep - best_ep >= a.patience:
            print(f"early stop: no val improvement for {a.patience} epochs (best epoch {best_ep + 1})")
            break

    json.dump({"params": n_params, "gmacs": gmacs, "best_val_mIoU_crops": best, "best_epoch": best_ep + 1,
               "epochs_run": ep + 1, "args": vars(a)},
              open(out / "summary.json", "w"), indent=2)
    print("done. best val mIoU(crops) =", best)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/agriseg/evaluate.py
"""Like-for-like comparison of AgriSegNet checkpoint(s) against YOLO26 segmentation weights.

    python evaluate.py --data /path/to/roboflow_export --split test --imgsz 640 \
        --custom runs/agri_s0/best.pt runs/agri_s1/best.pt runs/agri_s2/best.pt \
        --yolo yolo26n_seg_best.pt yolo26s_seg_best.pt --yolo-conf 0.25 --out results/compare

Fairness rules enforced here:
  * every model is scored on the SAME images at the ORIGINAL label resolution,
  * YOLO instance masks are converted to one semantic map (higher-confidence instance wins),
  * mIoU(crops) excludes background and is the headline number; mIoU(all) is also reported,
  * a paired bootstrap over test images gives a CI for (custom - yolo).
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader

from agrisegnet import count_flops, count_params
from dataset import CropSegEval, eval_collate
from engine import build_from_ckpt, predict_semantic
from metrics import ConfusionMatrix, instances_to_semantic

PALETTE = np.array([[0, 0, 0], [230, 25, 75], [60, 180, 75], [255, 225, 25], [0, 130, 200],
                    [245, 130, 48], [145, 30, 180], [70, 240, 240], [240, 50, 230]], np.uint8)


def img_miou_crops(cm: ConfusionMatrix):
    tp = np.diag(cm.m).astype(float)
    union = cm.m.sum(0) + cm.m.sum(1) - tp
    iou = np.where(union > 0, tp / np.maximum(union, 1), np.nan)
    return float(np.nanmean(iou[1:])) if np.any(~np.isnan(iou[1:])) else np.nan


def model_name(path) -> str:
    """Readable unique label: runs/agri_s0/best.pt -> agri_s0 ; runs/x/weights/best.pt -> x."""
    p = Path(path)
    d = p.parent.parent.name if p.parent.name == "weights" else p.parent.name
    return d if p.stem in ("best", "last") else f"{d}_{p.stem}" if d else p.stem


class CustomPredictor:
    def __init__(self, ckpt, device, amp, tta):
        self.model, self.ck = build_from_ckpt(ckpt, device)
        self.device, self.amp, self.tta = device, amp, tta
        self.name = model_name(ckpt)

    def __call__(self, batch):
        return predict_semantic(self.model, batch, self.device, self.amp, self.tta)


_YOLO_CACHE = {}


class YoloPredictor:
    def __init__(self, weights, imgsz, conf, device, tag_conf=False):
        from ultralytics import YOLO
        if weights not in _YOLO_CACHE:                 # one network in memory per weights file
            _YOLO_CACHE[weights] = YOLO(weights)
        self.m = _YOLO_CACHE[weights]
        self.imgsz, self.conf, self.device = imgsz, conf, device
        self.name = "yolo_" + model_name(weights) + (f"@conf{conf:g}" if tag_conf else "")

    def __call__(self, batch):
        h, w = batch["gt"].shape
        r = self.m.predict(batch["path"], imgsz=self.imgsz, conf=self.conf, device=self.device,
                           verbose=False, retina_masks=True)[0]
        if r.masks is None or len(r.masks) == 0:
            return np.zeros((h, w), np.uint8)
        return instances_to_semantic(r.masks.xy, r.boxes.cls.cpu().numpy().astype(int),
                                     r.boxes.conf.cpu().numpy(), h, w)


def bench_custom(pred: CustomPredictor, imgsz, device, n=100):
    m = pred.model
    x = torch.randn(1, 3, (imgsz + 127) // 128 * 128, (imgsz + 127) // 128 * 128, device=device)
    out = {}
    for tag, half in [("fp32", False), ("fp16", True)]:
        if half and device.type != "cuda":
            continue
        mm = m.half() if half else m.float()
        xx = x.half() if half else x.float()
        with torch.no_grad():
            for _ in range(20):
                mm(xx).argmax(1)
            if device.type == "cuda":
                torch.cuda.synchronize()
            t = time.perf_counter()
            for _ in range(n):
                mm(xx).argmax(1)
            if device.type == "cuda":
                torch.cuda.synchronize()
        out[f"latency_ms_{tag}"] = (time.perf_counter() - t) / n * 1000
    pred.model.float()
    return out


def bench_yolo(pred: YoloPredictor, sample_path, n=50):
    for _ in range(5):
        pred.m.predict(sample_path, imgsz=pred.imgsz, conf=pred.conf, device=pred.device, verbose=False)
    t = time.perf_counter()
    for _ in range(n):
        pred.m.predict(sample_path, imgsz=pred.imgsz, conf=pred.conf, device=pred.device, verbose=False)
    return {"latency_ms_e2e": (time.perf_counter() - t) / n * 1000}


def yolo_size(pred: YoloPredictor):
    try:
        pred.m.fuse()
        params = sum(p.numel() for p in pred.m.model.parameters())
        # verbose=True is required: with verbose=False ultralytics returns None (no counts)
        info = pred.m.info(verbose=True, imgsz=pred.imgsz)
        return {"params": params, "gflops": float(info[3]) if info and len(info) > 3 else None}
    except Exception as e:                                     # noqa: BLE001
        return {"params": None, "gflops": None, "note": str(e)}


def paired_bootstrap(a, b, n=2000, seed=0):
    a, b = np.asarray(a), np.asarray(b)
    k = ~np.isnan(a) & ~np.isnan(b)
    d = (a - b)[k]
    rng = np.random.default_rng(seed)
    means = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(n)])
    return float(d.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def colorize(m):
    return PALETTE[np.minimum(m, len(PALETTE) - 1)]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--split", default="test")
    ap.add_argument("--imgsz", type=int, default=512, help="inference size for AgriSegNet")
    ap.add_argument("--custom", nargs="*", default=[])
    ap.add_argument("--yolo", nargs="*", default=[])
    ap.add_argument("--yolo-imgsz", type=int, default=None, help="defaults to --imgsz")
    ap.add_argument("--yolo-conf", type=float, nargs="+", default=[0.25],
                    help="one or more confidence thresholds for the YOLO instance->semantic conversion. "
                         "Tune on --split valid, then report --split test at that single value.")
    ap.add_argument("--tta", action="store_true")
    ap.add_argument("--viz", type=int, default=12, help="number of qualitative montages to save")
    ap.add_argument("--no-bench", action="store_true")
    ap.add_argument("--out", default="results/compare")
    a = ap.parse_args()

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    yimgsz = a.yolo_imgsz or a.imgsz

    preds = [CustomPredictor(c, dev, amp=False, tta=a.tta) for c in a.custom]
    ydev = 0 if dev.type == "cuda" else "cpu"
    preds += [YoloPredictor(w, yimgsz, c, ydev, tag_conf=len(a.yolo_conf) > 1) for w in a.yolo for c in a.yolo_conf]
    if not preds:
        raise SystemExit("give at least one of --custom / --yolo")

    ds = CropSegEval(a.data, a.split, a.imgsz)
    names = ["background"] + list(ds.names)
    cms = {p.name: ConfusionMatrix(len(names)) for p in preds}
    per_img = {p.name: [] for p in preds}
    dl = DataLoader(ds, batch_size=1, shuffle=False, num_workers=2, collate_fn=eval_collate)

    for i, batch in enumerate(dl):
        gt = batch["gt"].numpy().astype(np.uint8)
        maps = {}
        for p in preds:
            pm = p(batch)
            maps[p.name] = pm
            cms[p.name].update(gt, pm)
            c1 = ConfusionMatrix(len(names)); c1.update(gt, pm)
            per_img[p.name].append(img_miou_crops(c1))
        if i < a.viz:
            tiles = [batch["rgb"], colorize(gt)] + [colorize(m) for m in maps.values()]
            h = 360
            tiles = [cv2.resize(t, (int(t.shape[1] * h / t.shape[0]), h), interpolation=cv2.INTER_NEAREST) for t in tiles]
            cv2.imwrite(str(out / f"viz_{i:03d}.jpg"), cv2.cvtColor(np.hstack(tiles), cv2.COLOR_RGB2BGR))

    rows = []
    for p in preds:
        s = cms[p.name].summary(names)
        row = {"model": p.name, "kind": "custom" if isinstance(p, CustomPredictor) else "yolo", **s}
        if isinstance(p, CustomPredictor):
            row["params"] = count_params(p.model)
            row["gflops"] = 2 * count_flops(p.model, (a.imgsz, a.imgsz))
            if not a.no_bench:
                row.update(bench_custom(p, a.imgsz, dev))
        else:
            row.update(yolo_size(p))
            if not a.no_bench and len(ds):
                row.update(bench_yolo(p, str(ds.images[0])))
        rows.append(row)

    # seed aggregation for the custom model
    cust = [r for r in rows if r["kind"] == "custom"]
    if len(cust) > 1:
        v = np.array([r["mIoU_crops"] for r in cust])
        print(f"\nAgriSegNet over {len(cust)} checkpoints: mIoU(crops) = {v.mean():.4f} +/- {v.std(ddof=1):.4f}")

    boot = {}
    ys = [p for p in preds if isinstance(p, YoloPredictor)]
    cs = [p for p in preds if isinstance(p, CustomPredictor)]
    for c in cs:
        for y in ys:
            boot[f"{c.name} - {y.name}"] = paired_bootstrap(per_img[c.name], per_img[y.name])

    json.dump({"rows": rows, "paired_bootstrap_mIoU_crops(mean,lo95,hi95)": boot, "classes": names,
               "args": vars(a)}, open(out / "results.json", "w"), indent=2)

    print(f"\n{'model':<28}{'params':>10}{'GFLOPs':>9}{'mIoU_crops':>12}{'mIoU_all':>10}{'pixAcc':>8}{'ms fp32/e2e':>13}{'ms fp16':>9}")
    for r in rows:
        ms = r.get("latency_ms_fp32", r.get("latency_ms_e2e"))
        print(f"{r['model']:<28}{(r.get('params') or 0):>10,}{(r.get('gflops') or 0):>9.2f}"
              f"{r['mIoU_crops']:>12.4f}{r['mIoU_all']:>10.4f}{r['pixel_acc']:>8.4f}"
              f"{(ms or 0):>13.1f}{(r.get('latency_ms_fp16') or 0):>9.1f}")
    print("\nper-class IoU:")
    for r in rows:
        print(f"  {r['model']:<26}", {k: (None if v is None else round(v, 3)) for k, v in r["per_class_iou"].items()})
    for k, (m, lo, hi) in boot.items():
        print(f"paired bootstrap  {k}: dmIoU = {m:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  (CI excluding 0 => real difference)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/agriseg/onnx_tools.py
"""Device-agnostic deployment checks: ONNX export (FP32), static INT8 quantisation (QDQ, per-channel) and
accuracy of the exported graphs with ONNX Runtime on a dataset split.

    python onnx_tools.py --ckpt runs/nano/best.pt --data <export> --imgsz 640 --out deploy/
writes deploy/model_fp32.onnx, deploy/model_int8.onnx and deploy/onnx_eval.json (mIoU FP32 vs INT8).

The QDQ INT8 graph is what TensorRT (Jetson), ONNX Runtime (Pi / x86 / ARM) and OpenVINO consume directly.
For microcontrollers convert the FP32 ONNX to TFLite INT8 (e.g. onnx2tf + TFLite converter) with the same
calibration images; that step is not automated here.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import torch

from dataset import CropSegEval, find_split_dir, IMG_EXT, preprocess
from engine import build_from_ckpt
from metrics import ConfusionMatrix


def export(model, path, size, output_stride=1):
    model.output_stride = output_stride if hasattr(model, "output_stride") else 1
    x = torch.zeros(1, 3, size, size)
    torch.onnx.export(model, x, str(path), opset_version=13, input_names=["image"], output_names=["logits"],
                      dynamic_axes=None, dynamo=False)
    model.output_stride = 1
    return path


def _square(x, size):
    """Pad a preprocessed CHW tensor to size x size (the exported graph has a fixed input shape)."""
    c, h, w = x.shape
    out = np.zeros((1, c, size, size), np.float32)
    out[0, :, :min(h, size), :min(w, size)] = x[:, :size, :size].numpy()
    return out


def calibration_reader(root, imgsz, size, n=100, seed=0):
    from onnxruntime.quantization import CalibrationDataReader
    import cv2
    imgs = sorted(p for p in (find_split_dir(root, "train") / "images").iterdir() if p.suffix.lower() in IMG_EXT)
    rng = np.random.default_rng(seed)
    pick = [imgs[i] for i in rng.choice(len(imgs), size=min(n, len(imgs)), replace=False)]

    class Reader(CalibrationDataReader):
        def __init__(self):
            self.it = iter(pick)

        def get_next(self):
            p = next(self.it, None)
            if p is None:
                return None
            x, _ = preprocess(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB), imgsz)
            return {"image": _square(x, size)}
    return Reader()


QUANT_OPS = ["Conv", "MatMul"]   # same INT8 recipe for every model compared: weights + inputs of Conv/MatMul


def export_size(arch, imgsz):
    m = 128 if arch == "agrisegnet" else 32
    return (imgsz + m - 1) // m * m


def quantize_int8(fp32, int8, reader):
    from onnxruntime.quantization import QuantFormat, QuantType, quantize_static
    from onnxruntime.quantization.shape_inference import quant_pre_process
    pre = Path(str(fp32).replace(".onnx", "_pre.onnx"))
    quant_pre_process(str(fp32), str(pre))
    quantize_static(str(pre), str(int8), reader, quant_format=QuantFormat.QDQ, per_channel=True,
                    activation_type=QuantType.QInt8, weight_type=QuantType.QInt8, op_types_to_quantize=QUANT_OPS)
    pre.unlink(missing_ok=True)
    import onnx                      # keep the FP32 model's metadata (Ultralytics reads task/names/imgsz from it)
    src, q = onnx.load(str(fp32)), onnx.load(str(int8))
    del q.metadata_props[:]
    q.metadata_props.extend(src.metadata_props)
    onnx.save(q, str(int8))
    return int8


def evaluate_onnx(path, root, split, imgsz, size, output_stride=1):
    import onnxruntime as ort
    import torch.nn.functional as F
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    ds = CropSegEval(root, split, imgsz)
    cm = ConfusionMatrix(ds.nc + 1)
    for i in range(len(ds)):
        b = ds[i]
        nh, nw = b["valid_hw"]
        logits = torch.from_numpy(sess.run(None, {"image": _square(b["image"], size)})[0])
        if output_stride != 1:
            logits = F.interpolate(logits, size=(size, size), mode="bilinear", align_corners=False)
        logits = F.interpolate(logits[:, :, :nh, :nw], size=tuple(b["gt"].shape), mode="bilinear", align_corners=False)
        cm.update(b["gt"].numpy().astype(np.uint8), logits.argmax(1)[0].numpy().astype(np.uint8))
    return cm.summary(["background"] + list(ds.names))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--split", default="test")
    ap.add_argument("--calib", type=int, default=100, help="number of train images for INT8 calibration")
    ap.add_argument("--output-stride", type=int, default=1, choices=[1, 8],
                    help="8: graph emits 1/8-resolution logits (upsample on the host); nano only")
    ap.add_argument("--out", default="deploy")
    a = ap.parse_args()
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    model, ck = build_from_ckpt(a.ckpt)
    size = export_size(ck["cfg"].get("arch", "agrisegnet"), a.imgsz)
    fp32 = export(model, out / "model_fp32.onnx", size, a.output_stride)
    int8 = quantize_int8(fp32, out / "model_int8.onnx", calibration_reader(a.data, a.imgsz, size, a.calib))
    res = {"input": [1, 3, size, size], "output_stride": a.output_stride}
    for tag, p in (("fp32", fp32), ("int8", int8)):
        r = evaluate_onnx(p, a.data, a.split, a.imgsz, size, a.output_stride)
        res[tag] = {"file_kib": round(p.stat().st_size / 1024, 1), **r}
        print(f"{tag}: {p.name} {res[tag]['file_kib']} KiB  mIoU(crops) {r['mIoU_crops']:.4f}  pixel acc {r['pixel_acc']:.4f}")
    res["int8_drop_mIoU_crops"] = res["fp32"]["mIoU_crops"] - res["int8"]["mIoU_crops"]
    json.dump(res, open(out / "onnx_eval.json", "w"), indent=2)
    print(f"INT8 mIoU drop: {res['int8_drop_mIoU_crops']:+.4f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/agriseg/benchmark.py
"""Deployment benchmark: every model x every input size x {FP32, INT8}, all run as ONNX on ONNX Runtime.

    python benchmark.py --data <export> --split test --sizes 640 480 320 \
        --custom runs/nano_s0/best.pt runs/nano_s1/best.pt --yolo yolo/yolo26n_scratch/weights/best.pt \
        --yolo-conf 0.25 --out results/benchmark

Same treatment for both model families, so the comparison is like for like:
  * export to a static ONNX graph at each input size;
  * INT8 = ONNX Runtime static quantisation, QDQ, per-channel, Conv/MatMul only, calibrated on the same
    `--calib` training images (each model with its own preprocessing);
  * accuracy = semantic mIoU on the split at the original label resolution (YOLO instances -> semantic map,
    highest confidence wins, conf fixed from the valid split);
  * latency = median ONNX Runtime CPU time of the graph alone (no pre/post-processing), `--threads` threads;
  * paired bootstrap 95 % CI of (custom - YOLO) per-image mIoU for each size x precision.
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F

from dataset import IMG_EXT, CropSegEval, find_split_dir, preprocess
from engine import build_from_ckpt
from evaluate import YoloPredictor, img_miou_crops, model_name, paired_bootstrap
from metrics import ConfusionMatrix
from onnx_tools import _square, export, export_size, quantize_int8


# ----------------------------------------------------------------------------- calibration readers
def _calib_images(root, n, seed=0):
    imgs = sorted(p for p in (find_split_dir(root, "train") / "images").iterdir() if p.suffix.lower() in IMG_EXT)
    idx = np.random.default_rng(seed).choice(len(imgs), size=min(n, len(imgs)), replace=False)
    return [imgs[i] for i in idx]


def _reader(images, feed):
    from onnxruntime.quantization import CalibrationDataReader

    class R(CalibrationDataReader):
        def __init__(self):
            self.it = iter(images)

        def get_next(self):
            p = next(self.it, None)
            return None if p is None else feed(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    return R()


def letterbox(rgb, size):
    """Ultralytics-style input: long side -> size, pad (114) to size x size, /255, NCHW."""
    h, w = rgb.shape[:2]
    r = size / max(h, w)
    im = cv2.resize(rgb, (round(w * r), round(h * r)), interpolation=cv2.INTER_LINEAR)
    c = np.full((size, size, 3), 114, np.uint8)
    top, left = (size - im.shape[0]) // 2, (size - im.shape[1]) // 2
    c[top:top + im.shape[0], left:left + im.shape[1]] = im
    return c.transpose(2, 0, 1)[None].astype(np.float32) / 255.0


# ----------------------------------------------------------------------------- per-model runners
class CustomOnnx:
    def __init__(self, path, imgsz, size, threads):
        import onnxruntime as ort
        so = ort.SessionOptions(); so.intra_op_num_threads = threads
        self.sess = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
        self.imgsz, self.size = imgsz, size
        self.feed_shape = (1, 3, size, size)

    def __call__(self, batch):
        nh, nw = batch["valid_hw"]
        logits = torch.from_numpy(self.sess.run(None, {"image": _square(batch["image"], self.size)})[0])
        if logits.shape[-1] != self.size:                       # output-stride 8 graphs
            logits = F.interpolate(logits, size=(self.size, self.size), mode="bilinear", align_corners=False)
        logits = F.interpolate(logits[:, :, :nh, :nw], size=tuple(batch["gt"].shape), mode="bilinear",
                               align_corners=False)
        return logits.argmax(1)[0].numpy().astype(np.uint8)

    def latency_ms(self, n=30):
        return _latency(self.sess, "image", self.feed_shape, n)


class YoloOnnx(YoloPredictor):
    def __init__(self, path, size, conf, threads):
        super().__init__(str(path), size, conf, "cpu")
        import onnxruntime as ort
        so = ort.SessionOptions(); so.intra_op_num_threads = threads
        self.sess = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
        self.feed_shape = (1, 3, size, size)

    def latency_ms(self, n=30):
        return _latency(self.sess, self.sess.get_inputs()[0].name, self.feed_shape, n)


def _latency(sess, name, shape, n):
    x = np.random.rand(*shape).astype(np.float32)
    for _ in range(5):
        sess.run(None, {name: x})
    t = []
    for _ in range(n):
        t0 = time.perf_counter(); sess.run(None, {name: x}); t.append(time.perf_counter() - t0)
    return float(np.median(t) * 1000)


# ----------------------------------------------------------------------------- exports
def export_custom(ckpt, size, out, calib):
    model, ck = build_from_ckpt(ckpt)
    s = export_size(ck["cfg"].get("arch", "agrisegnet"), size)
    fp32 = export(model, out / f"{model_name(ckpt)}_{s}_fp32.onnx", s)
    int8 = quantize_int8(fp32, out / f"{model_name(ckpt)}_{s}_int8.onnx",
                         _reader(calib, lambda rgb: {"image": _square(preprocess(rgb, size)[0], s)}))
    n_params = sum(p.numel() for p in model.parameters())
    return s, {"fp32": fp32, "int8": int8}, n_params


def export_yolo(weights, size, out, calib):
    from ultralytics import YOLO
    y = YOLO(str(weights))
    n_params = sum(p.numel() for p in y.model.parameters())
    tmp = Path(y.export(format="onnx", imgsz=size, simplify=True, dynamic=False, verbose=False))
    fp32 = out / f"{model_name(weights)}_{size}_fp32.onnx"
    tmp.replace(fp32)
    int8 = quantize_int8(fp32, out / f"{model_name(weights)}_{size}_int8.onnx",
                         _reader(calib, lambda rgb: {"images": letterbox(rgb, size)}))
    return size, {"fp32": fp32, "int8": int8}, n_params


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--split", default="test")
    ap.add_argument("--sizes", type=int, nargs="+", default=[640, 480, 320])
    ap.add_argument("--custom", nargs="*", default=[])
    ap.add_argument("--yolo", nargs="*", default=[])
    ap.add_argument("--yolo-conf", type=float, nargs="*", default=[0.25], help="one per --yolo, or one for all")
    ap.add_argument("--calib", type=int, default=100)
    ap.add_argument("--threads", type=int, default=1, help="ORT CPU threads for latency (1 = edge-like)")
    ap.add_argument("--out", default="results/benchmark")
    a = ap.parse_args()
    out = Path(a.out); (out / "onnx").mkdir(parents=True, exist_ok=True)
    calib = _calib_images(a.data, a.calib)
    confs = a.yolo_conf if len(a.yolo_conf) == len(a.yolo) else [a.yolo_conf[0]] * len(a.yolo)

    rows, boot = [], {}
    for size in a.sizes:
        ds = CropSegEval(a.data, a.split, size)
        names = ["background"] + list(ds.names)
        runners = []
        for ck in a.custom:
            s, files, n = export_custom(ck, size, out / "onnx", calib)
            for prec, f in files.items():
                runners.append((model_name(ck), "custom", prec, n, f, CustomOnnx(f, size, s, a.threads)))
        for w, c in zip(a.yolo, confs):
            s, files, n = export_yolo(w, size, out / "onnx", calib)
            for prec, f in files.items():
                runners.append((f"yolo_{model_name(w)}", "yolo", prec, n, f, YoloOnnx(f, s, c, a.threads)))

        cms = {r[0] + r[2]: ConfusionMatrix(len(names)) for r in runners}
        per_img = {k: [] for k in cms}
        for i in range(len(ds)):
            b = ds[i]
            gt = b["gt"].numpy().astype(np.uint8)
            for name, _, prec, _, _, run in runners:
                pm = run(b)
                cms[name + prec].update(gt, pm)
                one = ConfusionMatrix(len(names)); one.update(gt, pm)
                per_img[name + prec].append(img_miou_crops(one))
        for name, kind, prec, n, f, run in runners:
            r = cms[name + prec].summary(names)
            rows.append({"model": name, "kind": kind, "size": size, "precision": prec, "params": n,
                         "onnx_kib": round(f.stat().st_size / 1024, 1), "latency_ms_cpu": run.latency_ms(),
                         **r})
            print(f"{size:>4}px {prec} {name:<28s} mIoU(crops) {r['mIoU_crops']:.4f}  pix acc {r['pixel_acc']:.4f}  "
                  f"{rows[-1]['latency_ms_cpu']:.1f} ms")
        for prec in ("fp32", "int8"):
            for cn, ck_, cp, *_ in [r for r in runners if r[1] == "custom" and r[2] == prec]:
                for yn, yk, yp, *_ in [r for r in runners if r[1] == "yolo" and r[2] == prec]:
                    m, lo, hi = paired_bootstrap(per_img[cn + prec], per_img[yn + prec])
                    boot[f"{size}px {prec}: {cn} - {yn}"] = (m, lo, hi)
                    print(f"   {size}px {prec}: {cn} - {yn}: dmIoU {m:+.4f} [{lo:+.4f}, {hi:+.4f}]")

    json.dump({"rows": rows, "paired_bootstrap_mIoU_crops(mean,lo95,hi95)": boot, "args": vars(a)},
              open(out / "benchmark.json", "w"), indent=2)
    print(f"\n{'size':>4} {'prec':<5}{'model':<28}{'params':>11}{'ONNX KiB':>10}{'CPU ms':>8}{'mIoU crops':>12}")
    for r in sorted(rows, key=lambda r: (-r["size"], r["precision"], r["kind"], r["model"])):
        print(f"{r['size']:>4} {r['precision']:<5}{r['model']:<28}{r['params']:>11,}{r['onnx_kib']:>10.0f}"
              f"{r['latency_ms_cpu']:>8.1f}{r['mIoU_crops']:>12.4f}")


if __name__ == "__main__":
    main()

In [ ]:
# ---------------------------------------------------------------- configuration
# Roboflow export (YOLO segmentation format: train / valid / test with polygon labels).
# Key: preferably a Colab Secret named ROBOFLOW_API_KEY (key icon, left sidebar). You may paste it below instead,
# but then do not share this notebook: anyone with the file can use your Roboflow account.
ROBOFLOW_API_KEY = ''
RF_WORKSPACE, RF_PROJECT, RF_VERSION, RF_FORMAT = 'kavadis-workspace', 'crop_classification_orin', 4, 'yolo26'
DATASET_DIR = Path(os.environ.get('SEG_DATASET', str(ROOT / f'{RF_PROJECT}-{RF_VERSION}')))
RESULTS = Path(os.environ.get('SEG_OUT', str(RESULTS_ROOT / 'Segmentation')))

# Model and training (same budget for Nano and YOLO)
ARCH, WIDTH = 'nano', 1.0          # nano: 34.7k params (width 0.5 -> 11.4k)
TRAIN_IMGSZ = int(os.environ.get('SEG_IMGSZ', 640))
EPOCHS = int(os.environ.get('SEG_EPOCHS', 300))      # upper bound; early stopping ends training sooner
PATIENCE = int(os.environ.get('SEG_PATIENCE', 50))   # stop after this many epochs without val improvement (from-scratch starts slowly)
BATCH = 16
SEEDS = [0]                        # use [0, 1, 2] for any reported number (mean +/- std)
EVAL_SIZES = [int(x) for x in os.environ.get('SEG_EVAL_SIZES', '640 480 320').split()]   # multiples of 32
CALIB_IMAGES = 100                 # training images used to calibrate INT8 (same set for both models)

# Baseline: YOLO26n-seg from scratch (random init), like Nano
YOLO_BASELINES = {'yolo26n_scratch': 'yolo26n-seg.yaml'}
YOLO_CONF_SWEEP = [0.05, 0.1, 0.25, 0.4]                    # chosen on valid, then fixed for test

# Ground scale: Raspberry Pi Camera v2 (Sony IMX219, 8 MP, 3280 x 2464, 1.12 um pixels), flown at 25 m AGL.
# Footprint = H * sensor / focal = 30.26 m x 22.70 m; native GSD = 9.2 mm/px. The GSD of each dataset image is the
# footprint divided by that image's own width/height (valid only if Roboflow resized whole frames, not crops/tiles).
CAMERA = {'sensor_w_mm': 3.68, 'sensor_h_mm': 2.76, 'focal_mm': 3.04, 'native_px': (3280, 2464), 'height_agl_m': 25.0}
IMAGES_ARE_FULL_FRAMES = True      # Option A: Roboflow v4 'Resize 640x640' of full frames in every split
MIN_PATCH_FRACTION = 0.002         # connected regions smaller than this share of the image are ignored

# GLI (illustrative, NOT validated)
GLI_VEG_THRESHOLD = 0.0
GLI_THRESHOLDS = {'wheat': (0.05, 0.10, 0.15), 'potato': (0.04, 0.08, 0.13), 'gram': (0.03, 0.07, 0.12),
                  'fenugreek': (0.05, 0.09, 0.14), 'linseed': (0.05, 0.09, 0.14)}
REFERENCE_YIELD_KG_HA = {'wheat': 3547, 'potato': 24629, 'gram': 1199, 'fenugreek': 1582, 'linseed': 633}
YIELD_FACTORS = {'STRESSED / POOR': 0.50, 'MODERATE': 0.75, 'GOOD': 1.00, 'VERY GOOD': 1.10, 'UNKNOWN': np.nan}

PALETTE = [(0, 255, 255), (0, 255, 0), (255, 0, 255), (255, 255, 0), (0, 128, 255), (255, 0, 0), (0, 0, 255)]
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

sys.path.insert(0, str(AGRISEG_DIR))
from agrisegnet import build_model, count_params, count_flops, peak_activation_bytes   # noqa: E402
from engine import build_from_ckpt, predict_image                                      # noqa: E402
from dataset import load_names, find_split_dir                                          # noqa: E402


def run(cmd):
    """Run an agriseg script and stream its output into the notebook as it runs
    (a plain subprocess.run prints to Colab's background log, not to the cell)."""
    cmd = [sys.executable, '-u', *map(str, cmd)]
    print('$', ' '.join(cmd[2:]), flush=True)
    p = subprocess.Popen(cmd, cwd=AGRISEG_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)

In [ ]:
# ---------------------------------------------------------------- dataset
def download_dataset():
    if (DATASET_DIR / 'data.yaml').is_file():
        return DATASET_DIR
    key = ROBOFLOW_API_KEY or os.environ.get('ROBOFLOW_API_KEY')
    if not key and IN_COLAB:
        from google.colab import userdata
        try:
            key = userdata.get('ROBOFLOW_API_KEY')
        except Exception:                 # SecretNotFoundError / NotebookAccessError: fall through to the message below
            key = None
    if not key:
        raise RuntimeError('No Roboflow key: add the Colab Secret ROBOFLOW_API_KEY or fill ROBOFLOW_API_KEY above.')
    from roboflow import Roboflow
    ds = Roboflow(api_key=key).workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download(
        RF_FORMAT, location=str(DATASET_DIR))
    return Path(ds.location)


def dataset_yaml_abs(root):
    """Roboflow's data.yaml uses '../train/images'; Ultralytics needs paths it can resolve. Writes data_abs.yaml."""
    names = load_names(root)
    cfg = {'path': str(Path(root).resolve()), 'names': dict(enumerate(names))}
    for key, split in (('train', 'train'), ('val', 'valid'), ('test', 'test')):
        cfg[key] = str(find_split_dir(root, split).relative_to(root) / 'images')
    out = Path(root) / 'data_abs.yaml'
    yaml.safe_dump(cfg, open(out, 'w'), sort_keys=False)
    return out


def dataset_summary(root):
    names = load_names(root)
    print('classes:', names)
    all_sizes = {}
    for split in ('train', 'valid', 'test'):
        d = find_split_dir(root, split)
        imgs = sorted(p for p in (d / 'images').iterdir() if p.suffix.lower() in IMG_EXT)
        sizes = {}
        for p in imgs:
            h, w = cv2.imread(str(p)).shape[:2]
            sizes[f'{w}x{h}'] = sizes.get(f'{w}x{h}', 0) + 1
        print(f'{split:5s}: {len(imgs):4d} images, sizes {sizes}')
        all_sizes.update(sizes)
    # Option A: every split = full camera frame resized to one size, so train and test share the same GSD
    if len(all_sizes) != 1:
        print(f'WARNING: image sizes differ across/within splits {sorted(all_sizes)}: train and test GSD differ '
              '(crops or mixed preprocessing). Area per pixel below is only right for resized full frames.')
    else:
        w, h = map(int, next(iter(all_sizes)).split('x'))
        fw, fh = (CAMERA['height_agl_m'] * CAMERA[k] / CAMERA['focal_mm'] for k in ('sensor_w_mm', 'sensor_h_mm'))
        print(f'all splits {w}x{h}: GSD {100 * fw / w:.2f} x {100 * fh / h:.2f} cm/px '
              f'(native {1000 * fw / CAMERA["native_px"][0]:.2f} mm/px, {fw / w / (fw / CAMERA["native_px"][0]):.2f}x coarser)')
    return names


def dhash(path, size=8):
    g = cv2.resize(cv2.imread(str(path), cv2.IMREAD_GRAYSCALE), (size + 1, size), interpolation=cv2.INTER_AREA)
    return np.packbits((g[:, 1:] > g[:, :-1]).ravel())


def leakage_report(root, max_hamming=6):
    """Near-duplicate images across splits (overlapping UAV frames, augmentation copies). Any hit inflates test scores."""
    hashes = {}
    for s in ('train', 'valid', 'test'):
        d = find_split_dir(root, s) / 'images'
        hashes[s] = np.stack([dhash(p) for p in sorted(d.iterdir()) if p.suffix.lower() in IMG_EXT])
    for a, b in (('train', 'test'), ('train', 'valid'), ('valid', 'test')):
        dist = np.unpackbits(hashes[b][:, None, :] ^ hashes[a][None, :, :], axis=2).sum(2).min(1)
        print(f'{b} vs {a}: {int((dist <= max_hamming).sum())}/{len(dist)} near-duplicates '
              f'(median nearest Hamming distance {np.median(dist):.0f}/64)')


ds_root = download_dataset()
data_yaml = dataset_yaml_abs(ds_root)
class_names = dataset_summary(ds_root)
leakage_report(ds_root)

In [ ]:
# ---------------------------------------------------------------- model budget (no data needed)
_m = build_model(ARCH, len(class_names) + 1, WIDTH, aux=False)
print(f'{ARCH} width {WIDTH}: {count_params(_m):,} params, {count_flops(_m, (TRAIN_IMGSZ, TRAIN_IMGSZ)):.3f} GMACs @ {TRAIN_IMGSZ}')
for s in sorted({256, 320, TRAIN_IMGSZ}):
    print(f'  peak INT8 activations @ {s}px: {peak_activation_bytes(_m, (s, s)) / 2**20:.2f} MiB')

In [ ]:
# ---------------------------------------------------------------- train AgriSegNet-Nano (early stopping on valid mIoU)
RUNS = RESULTS / 'runs'
ckpts = []
for seed in SEEDS:
    out = RUNS / f'{ARCH}_w{WIDTH}_s{seed}'
    run(['train.py', '--data', ds_root, '--arch', ARCH, '--width', WIDTH, '--imgsz', TRAIN_IMGSZ,
         '--epochs', EPOCHS, '--patience', PATIENCE, '--bs', BATCH, '--seed', seed, '--workers', 2, '--out', out])
    ckpts.append(out / 'best.pt')
    print(json.load(open(out / 'summary.json'))['best_epoch'], 'best epoch of', json.load(open(out / 'summary.json'))['epochs_run'])

In [ ]:
# ---------------------------------------------------------------- train YOLO26n-seg from scratch (early stopping on valid)
from ultralytics import YOLO

yolo_ckpts = {}
for tag, init in YOLO_BASELINES.items():
    YOLO(init).train(data=str(data_yaml), imgsz=TRAIN_IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
                     seed=SEEDS[0], pretrained=False, project=str(RESULTS / 'yolo'), name=tag, exist_ok=True,
                     plots=False, device=0 if torch.cuda.is_available() else 'cpu', workers=2)
    yolo_ckpts[tag] = RESULTS / 'yolo' / tag / 'weights' / 'best.pt'
# Ultralytics stops on its own fitness (0.1 mAP50 + 0.9 mAP50-95 on valid); Nano stops on valid mIoU.

# native instance metrics (mask mAP) on test, for reference
yolo_native = {}
for tag, w in yolo_ckpts.items():
    m = YOLO(str(w)).val(data=str(data_yaml), split='test', imgsz=TRAIN_IMGSZ, plots=False, verbose=False)
    yolo_native[tag] = {'mask_mAP50': float(m.seg.map50), 'mask_mAP50_95': float(m.seg.map)}
print(json.dumps(yolo_native, indent=2))

In [ ]:
# ---------------------------------------------------------------- YOLO confidence threshold: chosen on VALID only
yolo_conf = {}
for tag, w in yolo_ckpts.items():
    out = RESULTS / 'yolo_conf_on_valid' / tag
    run(['evaluate.py', '--data', ds_root, '--split', 'valid', '--imgsz', TRAIN_IMGSZ, '--yolo', w,
         '--yolo-conf', *YOLO_CONF_SWEEP, '--no-bench', '--viz', 0, '--out', out])
    rows = json.load(open(out / 'results.json'))['rows']
    best = max(rows, key=lambda r: r['mIoU_crops'])
    yolo_conf[tag] = float(best['model'].rsplit('@conf', 1)[1])
    print(f'{tag}: conf {yolo_conf[tag]} (valid mIoU crops {best["mIoU_crops"]:.4f})')

In [ ]:
# ---------------------------------------------------------------- qualitative test montages (PyTorch, training size)
# image | ground truth | Nano | YOLO, saved as viz_*.jpg
for tag, w in yolo_ckpts.items():
    run(['evaluate.py', '--data', ds_root, '--split', 'test', '--imgsz', TRAIN_IMGSZ, '--custom', *ckpts,
         '--yolo', w, '--yolo-conf', yolo_conf[tag], '--viz', 8, '--out', RESULTS / 'montages_test' / tag])

In [ ]:
# ---------------------------------------------------------------- deployment benchmark on TEST: {640, 480, 320} x {FP32, INT8}
# Both models are exported to static ONNX at each size, INT8-quantised with the same recipe (ORT static QDQ, per-channel,
# Conv/MatMul, calibrated on the same training images) and scored with ONNX Runtime.
# CPU latency is ORT on the Colab CPU, 1 thread, graph only. INT8 latency on ORT-CPU is NOT representative (INT8 is
# emulated there and often slower); measure INT8 speed on the target (TensorRT on Jetson, TFLite on Pi/MCU).
BENCH = RESULTS / 'benchmark_test'
run(['benchmark.py', '--data', ds_root, '--split', 'test', '--sizes', *EVAL_SIZES, '--custom', *ckpts,
     '--yolo', *yolo_ckpts.values(), '--yolo-conf', *[yolo_conf[t] for t in yolo_ckpts],
     '--calib', CALIB_IMAGES, '--threads', 1, '--out', BENCH])
bench = json.load(open(BENCH / 'benchmark.json'))
for pair, (m, lo, hi) in bench['paired_bootstrap_mIoU_crops(mean,lo95,hi95)'].items():
    verdict = 'Nano better' if lo > 0 else 'YOLO better' if hi < 0 else 'no significant difference'
    print(f'{pair}: dmIoU {m:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}] -> {verdict}')
json.dump({'yolo_native_test': yolo_native, 'yolo_conf_from_valid': yolo_conf, 'benchmark': bench},
          open(RESULTS / 'comparison_summary.json', 'w'), indent=2)

In [ ]:
# ---------------------------------------------------------------- per-patch area and GLI from the custom model
def gsd_m(img_w, img_h):
    """Ground sample distance (m/px) along x and y for a full camera frame resized to img_w x img_h."""
    fw = CAMERA['height_agl_m'] * CAMERA['sensor_w_mm'] / CAMERA['focal_mm']
    fh = CAMERA['height_agl_m'] * CAMERA['sensor_h_mm'] / CAMERA['focal_mm']
    return fw / img_w, fh / img_h


def area_per_pixel_m2(img_w, img_h):
    if not IMAGES_ARE_FULL_FRAMES:
        return float('nan')
    gx, gy = gsd_m(img_w, img_h)
    return gx * gy


_fw, _fh = gsd_m(1, 1)
print(f"footprint {_fw:.2f} m x {_fh:.2f} m at {CAMERA['height_agl_m']} m AGL; native GSD "
      f"{1000 * _fw / CAMERA['native_px'][0]:.2f} mm/px")
for _w, _h in ((640, 640), (640, 480)):
    _gx, _gy = gsd_m(_w, _h)
    print(f'  {_w}x{_h} image: GSD {100 * _gx:.2f} x {100 * _gy:.2f} cm/px, {1e4 * _gx * _gy:.1f} cm2 per pixel'
          + ('  (stretched 4:3 -> 1:1: anisotropic but area is still correct)' if _w == _h else ''))


def gli_map(bgr):
    b, g, r = [bgr[..., i].astype(np.float32) for i in range(3)]
    den = 2 * g + r + b
    with np.errstate(divide='ignore', invalid='ignore'):
        out = (2 * g - r - b) / den
    out[den <= 0] = np.nan
    return out


def condition(cls, gli):
    if not np.isfinite(gli) or cls not in GLI_THRESHOLDS:
        return 'UNKNOWN'
    t1, t2, t3 = GLI_THRESHOLDS[cls]
    return 'STRESSED / POOR' if gli < t1 else 'MODERATE' if gli < t2 else 'GOOD' if gli < t3 else 'VERY GOOD'


def draw_label(img, text, x, y, placed):
    font, fs, th = cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1
    (tw, tht), base = cv2.getTextSize(text, font, fs, th)
    H, W = img.shape[:2]
    for dx, dy in ((-tw // 2, -10), (-tw // 2, tht + 20), (12, 0), (-tw - 12, 0)):
        x0 = int(np.clip(x + dx, 4, max(4, W - tw - 4))); y0 = int(np.clip(y + dy, tht + 4, H - base - 4))
        box = (x0 - 4, y0 - tht - 4, x0 + tw + 4, y0 + base + 4)
        if not any(not (box[2] < b[0] or b[2] < box[0] or box[3] < b[1] or b[3] < box[1]) for b in placed):
            break
    cv2.rectangle(img, box[:2], box[2:], (0, 0, 0), -1)
    cv2.putText(img, text, (x0, y0), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    placed.append(box)


def analyse_image(path, label_map, names):
    """label_map: 0 = background, c+1 = class c. Patches = connected regions of one class."""
    img = cv2.imread(str(path))
    h, w = img.shape[:2]
    gli, a_px = gli_map(img), area_per_pixel_m2(w, h)
    vis, placed, rows = img.copy(), [], []
    min_px = MIN_PATCH_FRACTION * h * w
    for c, name in enumerate(names, start=1):
        n, comp, stats, cent = cv2.connectedComponentsWithStats((label_map == c).astype(np.uint8), connectivity=8)
        for k in range(1, n):
            if stats[k, cv2.CC_STAT_AREA] < min_px:
                continue
            own = comp == k
            g = gli[own]; g = g[np.isfinite(g)]
            veg = g > GLI_VEG_THRESHOLD
            gli_all = float(g.mean()) if g.size else np.nan
            cond = condition(name, gli_all)
            area = int(own.sum()) * a_px
            rate = REFERENCE_YIELD_KG_HA.get(name, np.nan) / 1e4 * YIELD_FACTORS[cond]
            rows.append({'image': path.name, 'patch': len(rows) + 1, 'class': name, 'pixels': int(own.sum()),
                         'area_m2': area, 'veg_fraction': float(veg.mean()) if g.size else np.nan,
                         'gli_mean': gli_all, 'gli_median': float(np.median(g)) if g.size else np.nan,
                         'gli_mean_veg_only': float(g[veg].mean()) if veg.any() else np.nan,
                         'condition_indicative': cond, 'indicative_yield_kg': area * rate})
            col = np.array(PALETTE[(c - 1) % len(PALETTE)], np.float32)
            vis[own] = (vis[own].astype(np.float32) * 0.55 + col * 0.45).astype(np.uint8)
            draw_label(vis, f'#{len(rows)} {name} A:{area:.1f}m2 GLI:{gli_all:.2f}',
                       int(cent[k][0]), int(cent[k][1]), placed)
    # header: GSD and per-class totals for this image
    gx, gy = gsd_m(w, h)
    tot = {}
    for r in rows:
        tot[r['class']] = tot.get(r['class'], 0.0) + r['area_m2']
    lines = [f'GSD {100 * gx:.2f}x{100 * gy:.2f} cm/px @ {CAMERA["height_agl_m"]:.0f} m',
             '  '.join(f'{k}: {v:.1f} m2' for k, v in sorted(tot.items())) or 'no crop patches']
    bar = np.zeros((22 * len(lines) + 6, w, 3), np.uint8)
    for i, t in enumerate(lines):
        cv2.putText(bar, t, (6, 20 + 22 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
    return np.vstack([bar, vis]), rows


def run_patch_analysis(ckpt, image_dir, out_dir=RESULTS, imgsz=TRAIN_IMGSZ):
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model, ck = build_from_ckpt(ckpt, dev)
    names = ck['cfg']['names']
    (out_dir / 'overlays').mkdir(parents=True, exist_ok=True)
    rows = []
    for path in sorted(p for p in Path(image_dir).iterdir() if p.suffix.lower() in IMG_EXT):
        rgb = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        vis, rr = analyse_image(path, predict_image(model, rgb, imgsz, dev), names)
        cv2.imwrite(str(out_dir / 'overlays' / path.name), vis)
        rows += rr
    if rows:
        with open(out_dir / 'segmentation_patches.csv', 'w', newline='') as f:
            wr = csv.DictWriter(f, list(rows[0])); wr.writeheader(); wr.writerows(rows)
    with open(out_dir / 'SEGMENTATION_REPORT.txt', 'w') as f:
        f.write(f'AgriSegNet-Nano crop segmentation report\n{"=" * 70}\n')
        f.write(f'checkpoint {ckpt} | arch {ck["cfg"].get("arch")} | width {ck["cfg"]["width"]} | imgsz {imgsz}\n')
        f.write(f'camera {CAMERA} | full frames: {IMAGES_ARE_FULL_FRAMES}\n\n')
        f.write('Deployment benchmark on test (ONNX Runtime):\n')
        for r in sorted(bench['rows'], key=lambda r: (-r['size'], r['precision'], r['kind'], r['model'])):
            f.write(f"  {r['size']:>4}px {r['precision']} {r['model']:<26} params {r['params']:>10,}  "
                    f"mIoU(crops) {r['mIoU_crops']:.4f}  per class {r['per_class_iou']}\n")
        for pair, (m, lo, hi) in bench['paired_bootstrap_mIoU_crops(mean,lo95,hi95)'].items():
            f.write(f'  {pair}: dmIoU {m:+.4f} [{lo:+.4f}, {hi:+.4f}]\n')
        f.write('\n')
        f.write('Condition and yield are INDICATIVE (unvalidated thresholds and factors).\n\n')
        for name in sorted({r['class'] for r in rows}):
            rr = [r for r in rows if r['class'] == name]
            f.write(f'{name:<10s} patches {len(rr):3d}  area {sum(r["area_m2"] for r in rr):10.2f} m2  '
                    f'mean GLI {np.nanmean([r["gli_mean"] for r in rr]):.3f}\n')
        f.write(f'\nTotal area {sum(r["area_m2"] for r in rows):.2f} m2\n')
    print(f'{len(rows)} patches -> {out_dir}')
    return rows


seg_rows = run_patch_analysis(ckpts[0], find_split_dir(ds_root, 'test') / 'images')

In [ ]:
# ---------------------------------------------------------------- show segmented test images (area in m2 and GLI per patch)
import matplotlib.pyplot as plt
for f in sorted((RESULTS / 'overlays').iterdir())[:6]:
    im = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(9, 9 * im.shape[0] / im.shape[1])); plt.imshow(im); plt.title(f.name); plt.axis('off'); plt.show()

# Part 2: Sequoia calibration, alignment, NDVI and NDRE

# Parrot Sequoia: radiometric calibration, band alignment, NDVI and NDRE

Corrected version of `12_parrot_images (corrected).ipynb`.

**What was wrong in the previous version** (details in `docs/REVIEW.md`):

1. **Radiometry was a no-op.** The code asked exiftool for the tags `Black Level`, `Sensor Gain` and
   `Irradiance`. Sequoia files have none of them, so every band fell back to black = 64, gain = 1,
   irradiance = 1 (visible in the old output). The real dark level is about 4,300 to 4,600 DN (the
   `SensorModel` B coefficient) and band sensitivity differs about 3x between RED and REG
   (coefficient A). The old NDVI and NDRE were therefore ratios of raw DN. That is why mean NDRE
   came out *higher* than mean NDVI, which is physically backwards for vegetation.
2. **No vignetting correction.** Corners are 17 to 30 % darker than the centre (`VignettingPolynomial2D`).
3. Alignment bugs: affine composition in the wrong order (NIR -> REG -> RED), NCC scored on the
   zero-filled border, and failed captures fell back to *identity* (about 20 px misregistration)
   instead of the rig's known transform.
4. NDVI category thresholds (0.2 / 0.5) were reused for NDRE, which has a different range.
5. "Reflectance" CSVs were each band divided by its own maximum, which destroys the band ratios.
   They are removed.

**What this notebook does**

1. Reads the calibration metadata straight from each TIFF (XMP + EXIF), with no exiftool.
2. DN -> at-sensor irradiance with Parrot's model `I = f^2 (p - B) / (A*t*ISO + C)`, then divides by the
   vignetting polynomial.
3. Optional illumination normalisation from the Sunshine sensor, and optional reflectance-panel
   calibration.
4. Registers the bands (gradient images, phase-correlation start, coarse-to-fine ECC homography), checks
   every capture against the flight median, and falls back to the median transform when a capture fails.
5. Computes NDVI and NDRE on the common valid area and saves GeoTIFFs, PNGs, QC overlays and CSVs.

**Read before quoting numbers:** without a reflectance panel the indices are computed from
calibrated at-sensor irradiance, not surface reflectance. They are consistent within a flight but are
not comparable to literature thresholds. Photograph the panel before and after each flight and fill in
`PANEL` below.

In [ ]:
# ---------------------------------------------------------------- setup
import os, sys, re, json, base64, struct, subprocess, warnings
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rasterio', 'pillow', 'opencv-python-headless'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')

import numpy as np
import pandas as pd
import cv2
import rasterio
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ExifTags

warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)

In [ ]:
# ---------------------------------------------------------------- configuration
# The 12 TIFs (4 captures x NIR/RED/REG). A folder that was *shared with you* is not under MyDrive until you
# right-click it in Drive -> Organize -> Add shortcut -> My Drive. Then this path works.
SEQ_IMAGE_FOLDER = Path(os.environ.get('SEQUOIA_DIR', '/content/drive/MyDrive/parrot 12 images'))
# Outputs go to your own Drive (writing into someone else's shared folder can fail on permissions)
SEQ_OUT_ROOT = Path(os.environ.get('SEQUOIA_OUT', '/content/drive/MyDrive/AgriEdgeNet_Results/Sequoia_Indices'))
SEQ_EXPECTED_CAPTURES = 4

SEQ_BAND_SUFFIXES = {'GRE': ['GRE'], 'RED': ['RED'], 'REG': ['REG'], 'NIR': ['NIR']}
SEQ_BAND_ORDER = ['GRE', 'RED', 'REG', 'NIR']
SEQ_REFERENCE_BAND = 'REG'      # spectrally between RED and NIR: the best-correlated partner for both

# Radiometry
NORMALISE_ILLUMINATION = True   # scale each capture by Sunshine irradiance relative to the flight median
SUNSHINE_GAIN = {0: 1.0, 1: 4.0, 2: 16.0, 3: 64.0}   # gain index -> gain; only matters if indices differ
# Reflectance panel (set to None if no panel image was taken).
# 'capture' is the capture prefix of the panel shot, 'roi' = (row0, row1, col0, col1) INSIDE the panel
# in the aligned frame, 'reflectance' = the panel's certified reflectance per band.
PANEL = None
# PANEL = {'capture': 'IMG_260707_103000_0001', 'roi': (400, 520, 560, 700),
#          'reflectance': {'GRE': 0.18, 'RED': 0.18, 'REG': 0.18, 'NIR': 0.18}}

# Registration
PYRAMID_LEVELS = 4
ECC_MAX_ITER = 300
ECC_EPS = 1e-6
GRAD_BLUR_SIGMA = 1.5
MIN_ECC_CC = 0.30               # below this a capture is flagged
MAX_DEV_FROM_MEDIAN_PX = 3.0    # corner deviation from the flight-median transform before fallback

# Index category thresholds: illustrative only, calibrate against field data
INDEX_CLASSES = {
    'NDVI': (0.2, 0.5),
    'NDRE': (0.1, 0.3),
}

In [ ]:
# ---------------------------------------------------------------- metadata (XMP + EXIF, no exiftool)
def _xmp_text(path):
    raw = Path(path).read_bytes()
    s, e = raw.find(b'<x:xmpmeta'), raw.find(b'</x:xmpmeta>')
    if s < 0 or e < 0:
        raise ValueError(f'{path}: no XMP block; is this an original Sequoia TIFF?')
    return raw[s:e].decode('utf-8', 'replace')


def _xmp_attr(xmp, key):
    m = re.search(rf'Camera:{key}="([^"]*)"', xmp)
    return m.group(1) if m else None


def _xmp_seq(xmp, key):
    m = re.search(rf'<Camera:{key}>.*?<rdf:li>(.*?)</rdf:li>', xmp, re.S)
    return m.group(1).strip() if m else None


def _floats(s):
    return [float(v) for v in s.split(',')] if s else None


def read_sequoia_meta(path):
    """All calibration fields needed downstream. Raises if a required field is missing,
    instead of silently falling back to defaults (the old code's failure mode)."""
    xmp = _xmp_text(path)
    im = Image.open(path)
    exif = im.getexif()
    tags = {ExifTags.TAGS.get(k, k): v for k, v in list(exif.items()) + list(exif.get_ifd(0x8769).items())}

    sensor_model = _floats(_xmp_seq(xmp, 'SensorModel'))
    vig_pw = _xmp_seq(xmp, 'VignettingPolynomial2DName')
    vig_c = _floats(_xmp_seq(xmp, 'VignettingPolynomial2D'))
    meta = {
        'path': str(path),
        'band_name': _xmp_seq(xmp, 'BandName'),
        'exposure_s': float(tags.get('ExposureTime', 'nan')),
        'iso': float(tags.get('ISOSpeedRatings', 'nan')),
        'fnumber': float(tags.get('FNumber', 'nan')),
        'sensor_model': sensor_model,                     # A, B, C
        'vig_powers': [int(v) for v in vig_pw.split(',')] if vig_pw else None,
        'vig_coeffs': vig_c,
        'camera_ypr_deg': [_floats(_xmp_attr(xmp, k))[0] if _xmp_attr(xmp, k) else np.nan
                           for k in ('Yaw', 'Pitch', 'Roll')],
        'irradiance': None,
    }
    required = ['sensor_model', 'vig_powers', 'vig_coeffs']
    missing = [k for k in required if not meta[k]] + \
              [k for k in ('exposure_s', 'iso', 'fnumber') if not np.isfinite(meta[k])]
    if missing:
        raise ValueError(f'{Path(path).name}: missing calibration metadata {missing}')

    b64 = re.search(r'<Camera:IrradianceList>\s*(.*?)\s*</Camera:IrradianceList>', xmp, re.S)
    if b64:
        raw = base64.b64decode(b64.group(1).strip())
        recs = [struct.unpack('<QHHHH3f', raw[i:i + 28]) for i in range(0, len(raw) - 27, 28)]
        # (timestamp_us, ch0_count, ch1_count, gain_index, integration_ms, yaw, pitch, roll)
        if recs:
            meta['irradiance'] = {
                'gain_index': sorted({r[3] for r in recs}),
                'value': float(np.median([r[1] / (SUNSHINE_GAIN[r[3]] * r[4]) for r in recs])),
                'min_count': int(min(r[1] for r in recs)),
            }
    return meta

In [ ]:
# ---------------------------------------------------------------- radiometry
def vignetting_map(meta, shape):
    """Parrot 2-D vignetting polynomial on normalised coordinates x=col/W, y=row/H."""
    h, w = shape
    x = (np.arange(w, dtype=np.float64) / w)[None, :]
    y = (np.arange(h, dtype=np.float64) / h)[:, None]
    p = meta['vig_powers']
    v = np.zeros(shape, np.float64)
    for c, i, j in zip(meta['vig_coeffs'], p[0::2], p[1::2]):
        v += c * x ** i * y ** j
    return v.astype(np.float32)


def dn_to_sensor_irradiance(dn, meta):
    """Parrot Sequoia sensor model:  I = f^2 * (p - B) / (A * t * ISO + C), then / vignetting."""
    A, B, C = meta['sensor_model']
    f, t, iso = meta['fnumber'], meta['exposure_s'], meta['iso']
    I = f * f * (dn.astype(np.float32) - B) / (A * t * iso + C)
    return I / vignetting_map(meta, dn.shape)

In [ ]:
# ---------------------------------------------------------------- registration
def _unit(a):
    a = np.nan_to_num(a, nan=float(np.nanmedian(a)))
    lo, hi = np.percentile(a, (1, 99))
    return np.clip((a - lo) / (hi - lo + 1e-12), 0, 1).astype(np.float32)


def gradient_image(a):
    g = cv2.GaussianBlur(_unit(a), (0, 0), GRAD_BLUR_SIGMA)
    m = cv2.magnitude(cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3))
    return (m / (m.max() + 1e-12)).astype(np.float32)


def warp(img, W, shape, interp=cv2.INTER_LINEAR, border=0.0):
    h, w = shape
    return cv2.warpPerspective(img, W, (w, h), flags=interp | cv2.WARP_INVERSE_MAP,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=border)


def ncc(a, b, mask=None):
    if mask is not None:
        a, b = a[mask], b[mask]
    a = a - a.mean(); b = b - b.mean()
    return float((a * b).sum() / (np.sqrt((a * a).sum() * (b * b).sum()) + 1e-12))


def register(ref_g, mov_g):
    """Homography W with warp(mov, W) ~ ref. Phase-correlation start, coarse-to-fine ECC."""
    pr, pm = [ref_g], [mov_g]
    for _ in range(PYRAMID_LEVELS - 1):
        pr.append(cv2.pyrDown(pr[-1])); pm.append(cv2.pyrDown(pm[-1]))
    pr, pm = pr[::-1], pm[::-1]
    (dx, dy), _ = cv2.phaseCorrelate(pr[0], pm[0])
    cands = [np.eye(3, dtype=np.float32)]
    for s in (1.0, -1.0):
        Wc = np.eye(3, dtype=np.float32); Wc[0, 2], Wc[1, 2] = s * dx, s * dy
        cands.append(Wc)
    W = cands[int(np.argmax([ncc(pr[0], warp(pm[0], c, pr[0].shape)) for c in cands]))]
    crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, ECC_MAX_ITER, ECC_EPS)
    cc = float('nan')
    S = np.diag([0.5, 0.5, 1.0])
    for lvl, (r, m) in enumerate(zip(pr, pm)):
        try:
            cc, W = cv2.findTransformECC(r, m, W.astype(np.float32), cv2.MOTION_HOMOGRAPHY, crit, None, 5)
        except cv2.error:
            pass
        if lvl < len(pr) - 1:
            W = np.linalg.inv(S) @ W @ S
            W = (W / W[2, 2]).astype(np.float32)
    return W.astype(np.float32), float(cc)


def corner_deviation(W1, W2, shape):
    h, w = shape
    pts = np.array([[0, 0, 1], [w - 1, 0, 1], [0, h - 1, 1], [w - 1, h - 1, 1], [w / 2, h / 2, 1]], np.float64).T
    a, b = W1 @ pts, W2 @ pts
    return float(np.max(np.linalg.norm(a[:2] / a[2] - b[:2] / b[2], axis=0)))


def centre_shift(W, shape):
    h, w = shape
    p = W @ np.array([w / 2, h / 2, 1.0])
    return float(p[0] / p[2] - w / 2), float(p[1] / p[2] - h / 2)

In [ ]:
# ---------------------------------------------------------------- indices and output helpers
def normalized_difference(a, b, valid):
    with np.errstate(divide='ignore', invalid='ignore'):
        nd = (a - b) / (a + b)
    ok = valid & (a > 0) & (b > 0) & np.isfinite(nd)
    out = np.full(a.shape, np.nan, np.float32)
    out[ok] = nd[ok]
    return out


def class_breakdown(index, thresholds):
    v = index[np.isfinite(index)]
    lo, hi = thresholds
    n = max(v.size, 1)
    return {'low_%': 100 * np.sum(v < lo) / n, 'mid_%': 100 * np.sum((v >= lo) & (v <= hi)) / n,
            'high_%': 100 * np.sum(v > hi) / n}


def save_tif(path, arrays, names, tags=None):
    arrays = [np.asarray(a, np.float32) for a in arrays]
    h, w = arrays[0].shape
    with rasterio.open(path, 'w', driver='GTiff', height=h, width=w, count=len(arrays),
                       dtype='float32', nodata=np.nan, compress='deflate') as dst:
        for i, (a, n) in enumerate(zip(arrays, names), 1):
            dst.write(a, i); dst.set_band_description(i, n)
        if tags:
            dst.update_tags(**{k: str(v) for k, v in tags.items()})


def save_index_png(index, title, path, vmin, vmax):
    fig, ax = plt.subplots(figsize=(7, 5.4))
    cmap = matplotlib.colormaps['RdYlGn'].copy(); cmap.set_bad('black')
    im = ax.imshow(index, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title); ax.axis('off')
    fig.savefig(path, bbox_inches='tight', dpi=150); plt.close(fig)


def false_colour(bands, order):
    out = []
    for b in order:
        a = np.nan_to_num(bands[b], nan=0.0)
        lo, hi = np.nanpercentile(bands[b], (1, 99))
        out.append(np.clip((a - lo) / (hi - lo + 1e-12), 0, 1))
    return np.dstack(out)


def save_qc(name, before, after, path):
    order = ['NIR', 'RED', 'GRE' if 'GRE' in after else 'REG']
    fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
    ax[0].imshow(false_colour(before, order)); ax[0].set_title(f'{name}: before registration')
    ax[1].imshow(false_colour(after, order)); ax[1].set_title(f'{name}: registered (colour fringes = residual error)')
    for a in ax:
        a.axis('off')
    fig.savefig(path, bbox_inches='tight', dpi=120); plt.close(fig)

In [ ]:
# ---------------------------------------------------------------- pipeline
def find_capture_sets(folder):
    lut = {s.upper(): b for b, sufs in SEQ_BAND_SUFFIXES.items() for s in sufs}
    pat = re.compile(r'^(?P<prefix>.+?)_(?P<band>' + '|'.join(sorted(lut, key=len, reverse=True)) + r')$', re.I)
    sets = {}
    for p in sorted(Path(folder).iterdir()):
        if p.is_file() and p.suffix.lower() in {'.tif', '.tiff'}:
            m = pat.match(p.stem)
            if m:
                sets.setdefault(m.group('prefix'), {})[lut[m.group('band').upper()]] = p
    return sets


def calibrate(files):
    """-> metas, sensor-irradiance bands (float32), shape."""
    metas = {b: read_sequoia_meta(files[b]) for b in files}
    dn = {b: cv2.imread(str(files[b]), cv2.IMREAD_UNCHANGED) for b in files}
    shapes = {a.shape for a in dn.values()}
    if len(shapes) != 1:
        raise ValueError(f'band shapes differ: {shapes}')
    bands = {b: dn_to_sensor_irradiance(dn[b], metas[b]) for b in files}
    return metas, bands, shapes.pop()


def run_sequoia_pipeline(folder=None, out_root=None):
    folder = Path(folder or SEQ_IMAGE_FOLDER)
    out_root = Path(out_root or SEQ_OUT_ROOT)
    if not folder.is_dir():
        raise FileNotFoundError(f'{folder} not found. If the folder was shared with you, add a shortcut to '
                                'My Drive (Drive -> right-click -> Organize -> Add shortcut), or set SEQ_IMAGE_FOLDER.')
    dirs = {k: out_root / k for k in ('Calibrated_Bands', 'NDVI', 'NDRE', 'Alignment_QC')}
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)

    sets = find_capture_sets(folder)
    sets = {k: v for k, v in sets.items() if SEQ_REFERENCE_BAND in v and len(v) >= 2}
    if not sets:
        raise FileNotFoundError(f'No Sequoia capture sets with a {SEQ_REFERENCE_BAND} band in {folder}')
    print(f'{len(sets)} capture set(s) in {folder}')
    for k, v in sets.items():
        print(f'  {k}: {sorted(v)}')
    if len(sets) != SEQ_EXPECTED_CAPTURES:
        print(f'  WARNING: expected {SEQ_EXPECTED_CAPTURES} captures, found {len(sets)}')

    # ---- pass 1: radiometry and per-capture registration
    cap = {}
    for name, files in sets.items():
        metas, bands, shape = calibrate(files)
        ref_g = gradient_image(bands[SEQ_REFERENCE_BAND])
        reg = {}
        for b in bands:
            if b == SEQ_REFERENCE_BAND:
                continue
            W, cc = register(ref_g, gradient_image(bands[b]))
            reg[b] = {'W': W, 'cc': cc}
        cap[name] = {'files': files, 'metas': metas, 'bands': bands, 'shape': shape, 'reg': reg}
        print(f'  {name}: ' + ', '.join(f'{b} cc={r["cc"]:.3f}' for b, r in reg.items()))

    # ---- pass 2: rig consistency. The bands are rigidly mounted, so transforms should agree across
    # captures at similar altitude. Captures that disagree with the flight median use the median.
    bands_all = sorted({b for c in cap.values() for b in c['reg']})
    median_W = {}
    for b in bands_all:
        Ws = [c['reg'][b]['W'] for c in cap.values() if b in c['reg'] and c['reg'][b]['cc'] >= MIN_ECC_CC]
        if Ws:
            median_W[b] = np.median(np.stack(Ws), axis=0).astype(np.float32)

    # ---- illumination reference per band (flight median of Sunshine irradiance)
    sun_ref = {}
    for b in SEQ_BAND_ORDER:
        vals = [c['metas'][b]['irradiance']['value'] for c in cap.values()
                if b in c['metas'] and c['metas'][b]['irradiance']]
        if vals:
            sun_ref[b] = float(np.median(vals))
    gains = {b: c['metas'][b]['irradiance']['gain_index'] for c in cap.values() for b in c['metas']
             if c['metas'][b]['irradiance']}
    if len({tuple(g) for g in gains.values()}) > 1:
        print('  NOTE: Sunshine gain index varies; check SUNSHINE_GAIN mapping before trusting normalisation')

    align_rows, index_rows = [], []
    for name, c in cap.items():
        shape, bands = c['shape'], c['bands']
        ref_g = gradient_image(bands[SEQ_REFERENCE_BAND])
        aligned = {SEQ_REFERENCE_BAND: bands[SEQ_REFERENCE_BAND]}
        valid = np.isfinite(bands[SEQ_REFERENCE_BAND])
        for b, r in c['reg'].items():
            W, used = r['W'], 'ECC homography'
            dev = corner_deviation(W, median_W[b], shape) if b in median_W else float('nan')
            if r['cc'] < MIN_ECC_CC or (np.isfinite(dev) and dev > MAX_DEV_FROM_MEDIAN_PX):
                if b in median_W:
                    W, used = median_W[b], 'flight-median fallback'
                print(f'  {name} {b}: cc={r["cc"]:.3f}, deviation {dev:.1f} px -> {used}')
            mov_g = gradient_image(bands[b])
            out = warp(bands[b], W, shape, border=np.nan)
            ok = np.isfinite(out)
            ncc_before = ncc(ref_g, mov_g, valid & np.isfinite(bands[b]))
            ncc_after = ncc(ref_g, warp(mov_g, W, shape), valid & ok)
            aligned[b] = out
            valid &= ok
            sx, sy = centre_shift(W, shape)
            align_rows.append({'capture': name, 'band': b, 'reference': SEQ_REFERENCE_BAND, 'method': used,
                               'ecc_cc': r['cc'], 'grad_ncc_before': ncc_before, 'grad_ncc_after': ncc_after,
                               'centre_shift_x_px': sx, 'centre_shift_y_px': sy,
                               'dev_from_flight_median_px': dev, 'homography': json.dumps(np.round(W, 6).tolist())})
        # common valid area -> NaN elsewhere (no cropping, so every capture keeps the same pixel grid)
        for b in aligned:
            aligned[b] = np.where(valid, aligned[b], np.nan).astype(np.float32)

        # illumination normalisation (Sunshine), relative to the flight median per band
        level = 'sensor_irradiance'
        if NORMALISE_ILLUMINATION and all(b in sun_ref and c['metas'][b]['irradiance'] for b in aligned):
            for b in aligned:
                aligned[b] = aligned[b] * (sun_ref[b] / c['metas'][b]['irradiance']['value'])
            level = 'sensor_irradiance_sun_normalised'
        c['aligned'], c['valid'], c['level'] = aligned, valid, level

    # ---- optional panel calibration -> reflectance
    K_panel = None
    if PANEL:
        pc = cap.get(PANEL['capture'])
        if pc is None:
            raise KeyError(f'panel capture {PANEL["capture"]} not found')
        r0, r1, c0, c1 = PANEL['roi']
        K_panel = {b: PANEL['reflectance'][b] / float(np.nanmean(pc['aligned'][b][r0:r1, c0:c1]))
                   for b in pc['aligned']}
        print('  panel factors:', {b: f'{k:.4g}' for b, k in K_panel.items()})

    for name, c in cap.items():
        aligned, valid, level = c['aligned'], c['valid'], c['level']
        if K_panel:
            aligned = {b: a * K_panel[b] for b, a in aligned.items()}
            level = 'reflectance_panel'
        names = [b for b in SEQ_BAND_ORDER if b in aligned]
        save_tif(dirs['Calibrated_Bands'] / f'{name}_bands.tif', [aligned[b] for b in names], names,
                 {'radiometric_level': level, 'reference_band': SEQ_REFERENCE_BAND})
        save_qc(name, c['bands'], aligned, dirs['Alignment_QC'] / f'{name}_alignment_QC.png')

        row = {'capture': name, 'radiometric_level': level, 'valid_area_%': 100 * valid.mean()}
        idx = {}
        if 'RED' in aligned and 'NIR' in aligned:
            idx['NDVI'] = normalized_difference(aligned['NIR'], aligned['RED'], valid)
        if 'REG' in aligned and 'NIR' in aligned:
            idx['NDRE'] = normalized_difference(aligned['NIR'], aligned['REG'], valid)
        for k, a in idx.items():
            save_tif(dirs[k] / f'{name}_{k}.tif', [a], [k], {'radiometric_level': level})
            save_index_png(a, f'{name} {k} ({level})', dirs[k] / f'{name}_{k}.png',
                           -0.2 if k == 'NDVI' else -0.1, 1.0 if k == 'NDVI' else 0.6)
            v = a[np.isfinite(a)]
            row.update({f'{k}_mean': float(v.mean()), f'{k}_median': float(np.median(v)),
                        f'{k}_p05': float(np.percentile(v, 5)), f'{k}_p95': float(np.percentile(v, 95))})
            row.update({f'{k}_{kk}': vv for kk, vv in class_breakdown(a, INDEX_CLASSES[k]).items()})
        if {'NDVI', 'NDRE'} <= idx.keys():
            veg = idx['NDVI'] > 0.4
            row['NDRE_gt_NDVI_on_veg_%'] = float(100 * np.mean(idx['NDRE'][veg] > idx['NDVI'][veg])) if veg.any() else np.nan
            if row['NDRE_gt_NDVI_on_veg_%'] > 5:
                print(f'  WARNING {name}: NDRE > NDVI on {row["NDRE_gt_NDVI_on_veg_%"]:.1f}% of vegetation '
                      'pixels; radiometry is probably wrong')
        index_rows.append(row)
        c['indices'] = idx

    pd.DataFrame(align_rows).to_csv(out_root / 'alignment_summary.csv', index=False)
    summary = pd.DataFrame(index_rows)
    summary.to_csv(out_root / 'indices_summary.csv', index=False)
    with open(out_root / 'PROCESSING_REPORT.txt', 'w') as f:
        f.write('PARROT SEQUOIA: CALIBRATED + ALIGNED NDVI / NDRE\n' + '=' * 70 + '\n')
        f.write(f'Input folder        : {folder}\nReference band      : {SEQ_REFERENCE_BAND}\n')
        f.write('Radiometry          : I = f^2 (DN - B) / (A t ISO + C) / V(x,y)   [Parrot SensorModel, vignetting]\n')
        f.write(f'Illumination        : {"Sunshine-normalised to flight median" if NORMALISE_ILLUMINATION else "none"}\n')
        f.write(f'Panel calibration   : {"yes" if K_panel else "NO -> indices are not surface reflectance"}\n')
        f.write('Geometry            : ECC homography per band on gradient images (raw frames)\n\n')
        f.write(pd.DataFrame(align_rows).drop(columns='homography').to_string(index=False) + '\n\n')
        f.write(summary.to_string(index=False) + '\n')
    print(summary.round(3).to_string(index=False))
    print(f'\nOutputs: {out_root}')
    return cap, summary

In [ ]:
if __name__ == '__main__' or IN_COLAB:
    if os.environ.get('SEQ_SKIP_RUN') != '1':
        seq_captures, seq_summary = run_sequoia_pipeline()

## Package results

In [ ]:
import shutil
bundle = Path('/content/Combined_Results') if IN_COLAB else Path('Combined_Results')
bundle.mkdir(exist_ok=True)
for src, name in ((RESULTS, 'Segmentation'), (SEQ_OUT_ROOT, 'Sequoia_Indices')):
    if Path(src).exists():
        shutil.copytree(src, bundle / name, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns('*.pt', '*.cache', 'runs', 'yolo'))   # weights stay on Drive
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print('archive:', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)